# Account-Level Loss Compression for Marginal Impact Analysis

Implementation of *Account-Level Loss Compression for Marginal Impact Analysis* (31 July 2026) on the
PLT schema in use:

| table | columns | grain |
|---|---|---|
| `account_plt` | `accnt_no, event_id, year_id, loss_date, loss` | one row per account per occurrence |
| `portfolio_plt` | `event_id, year_id, loss_date, loss` | one row per occurrence |

Occurrence key `(year_id, event_id, loss_date)`: a year holds many loss dates, a date holds many
accounts, and one `event_id` may recur in a year on different dates.

## Seven corrections to the document

1. **`α ≤ q` is inverted (§4.2, §7.2).** Retention keeps the top `(1−q)T` years; the allocation reads
   the top `(1−α)T`. Containment needs `(1−α) ≤ (1−q)`, i.e. **`α ≥ q`**. §4.1.3's prose is correct;
   the two formal statements contradict it. Shipped as `assert alpha <= q` this blocks the correct
   configuration and passes the unsafe one.
2. **§5.1's covariance is the *population* covariance (÷n).** `np.cov` defaults to `ddof=1` and breaks
   the identity by `n/(n−1)`.
3. **Test 1 is an implementation test, not an adequacy test.** Aggregate-then-divide makes per-account
   AAL exact for *any* partition, one degenerate cell included. Non-zero means unweighted shares or
   bands calibrated on the full table — never "bands too coarse".
4. **A fourth test is needed.** With tests 1, 2 and additivity all exact by construction, nothing in
   §8 grades the bands. Added: per-body-row reconstruction error.
5. **§7.1's illustration is arithmetically wrong.** It states `s_C = 12/37`, but C's column is
   1+1+11 = 13; the tell is that 11/37 + 13/37 + 12/37 = 36/37 and shares must sum to 1. Multiplier is
   59/37 = 1.5946, not 58/37; rows are 15.95 / 19.14 / 23.92, not 15.7 / 18.8 / 23.5. **The conclusion
   is untouched** — both orderings are exactly as stated.
6. **§9's sizing formula under-predicts structurally.** It uses book-average events/year, but retained
   years are selected *for* annual severity, which correlates with occurrence count. Corrected to
   condition on `E[events | year ∈ R]`.
7. **Band resolution has far less leverage than §5.2 and §8-test-3 imply.** The body error is dominated
   by independent secondary uncertainty about the cell's conditional mean — the term §7.2 says is
   discarded, which no banding can remove.

## Parallelism and chunking (§10.1, §10.2)

- **Chunk grid is fixed, worker count is not part of run identity.** The document says worker count
  must be recorded because it sets the chunk boundaries. Fixing `CHUNK_YEARS` instead makes the book
  independent of how many cores ran it — asserted below at 1, 2 and 4 workers.
- **Shares are reduced, not averaged.** Workers return `(Σ Lₐ,ᵣ, Σ Lᵣ)` per cell; the division happens
  once after the reduction. Averaging per-worker shares would reintroduce the §5.1 unweighted bias at
  worker granularity.
- **`np.add.at` / `np.maximum.at`** for the Y/X accumulators — correct under any partition, not just
  a partition by year.
- **Band edges are *a priori*,** derived from the ELTs' analytic mean losses rather than body
  quantiles. This removes the second global barrier: only the retention threshold now needs one.
- **Peak memory is genuinely O(chunk)** on both production passes. The full account table is
  materialised only in the separately-labelled §8 validation run.

## Revision 2 — code-review corrections

Distinct from the seven *document* corrections above, these fix or harden the revision-1
**implementation**. All identities and conclusions are unchanged.

1. **R1 — `severity_hash` is now a content hash.** The old hash concatenated `ELTS.values()` with no
   account label: swapping which account owns which ELT (insertion order preserved) *collided*, and
   an identical book built in a different SPEC order *false-alarmed*.
2. **R2 — the audit record can now support its CRN claim.** Bit-reproducibility is only defined at
   fixed library versions and a fixed RNG consumption order; the audit now pins python/numpy/pandas/
   scipy versions, the platform, a generator version, and a hash of the worker source.
3. **R3 — `reconstruct_body` fails loudly** on cells with no body observations instead of emitting
   silent NaN allocated losses (reachable when reconstructing new portfolio tables downstream).
4. **R4 — the CRN check compares every column exactly**, not just `loss`.
5. **R5 — book-dependent assertions are gated on measured properties** (EXPECT_FAIL containment,
   §9 bias ordering, §9 sizing tolerance). Identities stay hard asserts. §6's rule, applied to
   the battery itself: measure, do not assume.
6. **R6 — the harness verdict's T2 tolerance is relative** to the co-TVaR scale (`tol2` was
   computed and unused while a hard-coded absolute `1e-6` did the work).
7. **R7 — the generator is vectorised across events**: one `(E, T)` Poisson draw and one
   array-parameter `beta.ppf` call per account per chunk. Cost previously scaled with catalogue
   size (measured 12× overhead at 4,000 events; real ELTs run 10⁴–10⁵). **The RNG layout changes:
   revision-2 books are statistically identical but not bit-identical to revision 1** — which is
   exactly why R2 versions the generator inside the audit.
8. **R8 — integer cell codes** (`peril_code·1000 + band`) replace per-row f-strings in the hot
   paths; labels are rebuilt only for display and audit.
9. **R9 — shared state ships once per worker** via `ProcessPoolExecutor(initializer=...)`; a task
   is now just `(year_lo, year_hi, seed)`. Per-task payload pickling is why 1 worker beat 4 at
   demo scale.
10. **R10 — guards**: per-event annual count ≤ 365 (protects occurrence-key uniqueness), all-empty
    reduction, invariance reruns behind `RUN_INVARIANCE_CHECKS`, version-agnostic groupby-apply,
    two-sided share-matrix comparison.
11. **R11 (documentation)** — the appendix claimed unweighted shares break the per-cell sum-to-1
    invariant. False: densified unweighted shares sum to 1 *exactly*, because per-row shares do
    (Σₐ sₐ,ᵣ = 1). Unweighted is caught by **test 1 alone** — which is precisely correction 3's
    point about what test 1 measures.

## Revision 3 — envelope retention: the contracted marginal-impact read

Revision 2’s §13 guard *prohibited* every non-uniform re-weighting — the tool’s titular query.
Revision 3 inverts the design: declare the perturbation family up front and retain enough years
that re-weighted allocation is **exact for every admissible weighting**, read from the tail store
alone.

- **E1** — pass 1 streams two extra reducible accumulators per grain: `Low(t) = Σₐ w_lo,ₐ Lₐ,ₜ`
  and `Env(t) = Σₐ w_hi,ₐ Lₐ,ₜ` (sums for AEP, per-occurrence-then-max for OEP). O(T) memory,
  `np.add.at`/`np.maximum.at`, any partition.
- **E2** — retain every year with `Env(t) ≥ τ`, τ = m-th largest `Low`. One-line containment
  proof (§6): for `v ∈ W`, `Low ≤ Y_v ≤ Env` pointwise, so an α-tail year under `v` has
  `Env(t) ≥ Y_v(t) ≥ m`-th largest of `Y_v` `≥ τ`. Uniform scaling never re-ranks, so the
  supported set is the whole **cone** `{c·v : c > 0, v ∈ W}`. With `W = {1}` this degenerates
  to the base scheme.
- **E3** — the guard becomes three-zone: uniform (any α ≥ q, scale the base answer) · inside the
  cone (`maxₐ(wₐ/hiₐ) ≤ minₐ(wₐ/loₐ)`: exact at α ≥ ALPHA) · outside (blocked, with a
  widen-W-or-regenerate message). The bounds live in the audit record.
- **E4** — validated by sampling `w` across interior, corners and cone points against the full CRN
  account table: containment, exactness (≤ 1e-9 rel) and additivity asserted per sample; the price
  (extra retained years) is measured and printed, and the certification battery carries all three.
- **E5** — harness axes: `envelope` (box/none, with a rev-2 preset showing what the price buys),
  `band_basis="event"` (correction 7 pushed to one cell per event — finest partition, T1 still
  exact), and `share_estimator="analytic"` (shares from ELT MeanLoss: zero fitting, zero barrier,
  **declared trade**: gives up exact per-account AAL, so it is expected-REJECTED under the strict
  contract).
- **E6** — deferred design notes recorded in the Summary: an annual-sum middle tier for the body,
  and a single-pass a-priori retention threshold for vendor-sourced PLTs.

**Revision 3.1 (post-review).** Independent review verified E1–E4 and surfaced a structural
consequence now documented in §14: the envelope floor (τ from ALPHA, not q) makes §9 reductions
1–2 inert — measured as exact set-collapse onto `R_ENV` — leaving grain as the only remaining
size lever. Added: the closed-form W-width price sweep (plateau → knee → cliff), the
locality-principle note in §13, a sharper reading of the 3× measurement, and single-pass
a-priori retention elevated from design note to next work item.

In [ ]:
import numpy as np, pandas as pd, scipy, hashlib, json, platform, shutil, sys, importlib, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from scipy import stats

# ---- run configuration ----------------------------------------------------
ROOT_SEED   = 20260731
T_YEARS     = 40_000
CHUNK_YEARS = 5_000          # fixes the chunk grid -> worker count is NOT part of run identity
N_WORKERS   = 4              # free parameter: changing it must not change the book
Q_RETAIN    = 0.975          # retention percentile
ALPHA       = 0.99           # working exceedance level
N_BANDS     = 8              # loss bands per peril group

# ---- E1: marginal-impact envelope ------------------------------------------
# The contracted perturbation family: per-account weight bounds W = prod [w_lo, w_hi].
# Re-weighted allocation is EXACT from the store for every w in the cone
# {c*v : c > 0, v in W} at alpha >= ALPHA (proof in section 6). Scalars here for the
# demo; per-account dicts are what actually travel to the workers, so heterogeneous
# bounds slot straight in.
W_LO, W_HI = 0.85, 1.20      # supported weight-SPREAD contract: max(w)/min(w) <= 1.20/0.85

RUN_INVARIANCE_CHECKS = True # R10: sections 5 and 7 rerun generation to prove invariance;
                             # switch off for large T once the property is established

OCC_KEY  = ["year_id", "event_id", "loss_date"]
ACC_COLS = ["accnt_no"] + OCC_KEY + ["loss"]

STORE = Path("compressed_store")
if STORE.exists():
    shutil.rmtree(STORE)
(STORE / "tail").mkdir(parents=True)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# --- CORRECTION 1: containment requires alpha >= q, not alpha <= q ---------
assert ALPHA >= Q_RETAIN, (
    f"retention at q={Q_RETAIN} keeps the top {1-Q_RETAIN:.1%} of years; allocation at "
    f"alpha={ALPHA} needs the top {1-ALPHA:.1%} - not contained")
assert 0 < W_LO <= 1.0 <= W_HI, "the base book (w = 1) must sit inside the envelope"
N_CHUNKS = int(np.ceil(T_YEARS / CHUNK_YEARS))
print(f"T = {T_YEARS:,} years | chunk grid = {CHUNK_YEARS:,} years -> {N_CHUNKS} chunks")
print(f"q = {Q_RETAIN} | alpha = {ALPHA} | workers = {N_WORKERS}")
print(f"containment: top {1-ALPHA:.1%} (allocation) subset of top {1-Q_RETAIN:.1%} (retention) OK")
print(f"envelope W = [{W_LO}, {W_HI}] per account -> supported spread {W_HI/W_LO:.3f}")

## 1 · The book — event catalogue and per-account ELTs

Eleven accounts over three peril groups; `FL-03` and `FL-04` are multi-peril, so the share matrix is
genuinely sparse. Ladder slopes differ by account on purpose: a concentrated coastal account's damage
ratio climbs far faster with event severity than a diffuse inland one. That is what makes an account's
share correlate with occurrence size — the effect the loss-weighted estimator exists to capture (§5.1).

In [ ]:
def geo_ladder(lo, hi, m):
    return lo * (hi / lo) ** (np.arange(m) / (m - 1))

def make_elt(event_ids, rates, mdr, exposure, cv_indep, cv_corr):
    ml = np.asarray(mdr) * exposure
    return pd.DataFrame({"EventId": np.asarray(event_ids), "Rate": np.asarray(rates),
                         "MeanLoss": ml, "SdIndep": cv_indep * ml, "SdCorr": cv_corr * ml,
                         "Exposure": float(exposure)})

FL = np.arange(1000, 1020); FL_RATE = 0.25 * 0.82 ** np.arange(20)
SC = np.arange(2000, 2010); SC_RATE = 0.30 * 0.85 ** np.arange(10)
JP = np.arange(3000, 3010); JP_RATE = 0.20 * 0.75 ** np.arange(10)

catalogue = pd.concat([
    pd.DataFrame({"event_id": FL, "Rate": FL_RATE, "peril_group": "FL_WIND"}),
    pd.DataFrame({"event_id": SC, "Rate": SC_RATE, "peril_group": "SE_CONV"}),
    pd.DataFrame({"event_id": JP, "Rate": JP_RATE, "peril_group": "JP_QUAKE"}),
], ignore_index=True)

#          account   peril      lo     hi    expo  cv_ind cv_corr
SPEC = [("FL-01", "FL_WIND",  0.002, 0.40,   800, 0.55, 0.35),   # coastal, concentrated
        ("FL-02", "FL_WIND",  0.003, 0.34,   600, 0.55, 0.35),
        ("FL-03", "FL_WIND",  0.004, 0.18,  1200, 0.50, 0.30),   # mixed
        ("FL-04", "FL_WIND",  0.006, 0.06,  1500, 0.45, 0.25),   # inland, diffuse
        ("FL-05", "FL_WIND",  0.005, 0.05,   900, 0.45, 0.25),
        ("FL-06", "FL_WIND",  0.002, 0.38,   300, 0.55, 0.35),
        ("SC-01", "SE_CONV",  0.002, 0.040,  700, 0.60, 0.25),
        ("SC-02", "SE_CONV",  0.003, 0.030, 1100, 0.60, 0.25),
        ("FL-03", "SE_CONV",  0.002, 0.035, 1200, 0.55, 0.25),   # multi-peril
        ("FL-04", "SE_CONV",  0.004, 0.020, 1500, 0.50, 0.20),   # multi-peril
        ("JP-01", "JP_QUAKE", 0.010, 0.30,  1000, 0.50, 0.40),
        ("JP-02", "JP_QUAKE", 0.008, 0.15,  1400, 0.45, 0.35),
        ("JP-03", "JP_QUAKE", 0.012, 0.35,   500, 0.50, 0.40)]

IDS = {"FL_WIND": FL, "SE_CONV": SC, "JP_QUAKE": JP}
RTS = {"FL_WIND": FL_RATE, "SE_CONV": SC_RATE, "JP_QUAKE": JP_RATE}

ELTS = {}
for acct, peril, lo, hi, expo, cvi, cvc in SPEC:
    ids = IDS[peril]
    elt = make_elt(ids, RTS[peril], geo_ladder(lo, hi, len(ids)), expo, cvi, cvc)
    ELTS[acct] = pd.concat([ELTS[acct], elt], ignore_index=True) if acct in ELTS else elt

ACCOUNTS = sorted(ELTS)
PERIL_OF = dict(zip(catalogue.event_id.astype(int), catalogue.peril_group))
aal_true = pd.Series({a: (e.Rate * e.MeanLoss).sum() for a, e in ELTS.items()}, name="AAL analytic")

print(f"{len(ACCOUNTS)} accounts, {len(catalogue)} events, "
      f"{catalogue.Rate.sum():.2f} expected occurrences/year")
display(ELTS["FL-01"].head(4))
display(aal_true.loc[ACCOUNTS].to_frame())

## 2 · A priori band edges — removing the second global barrier

The document keys cells on the row's own loss × peril group (§5.2), which is right. But calibrating
band *edges* on body quantiles forces a second global barrier: no worker can assign a cell until every
worker has finished. Cell definitions only have to be **deterministic**, not data-optimal — the
aggregate-then-divide identity and per-account AAL exactness hold for any partition.

So edges are log-spaced over the portfolio-level analytic mean loss per event, known from the ELTs
before a single year is simulated. Cost: slightly less balanced cells. Given correction 7 (band
resolution buys little), that is a cheap trade for removing a barrier.

In [ ]:
ev_mean = {}                                    # event_id -> portfolio-level mean loss
for elt in ELTS.values():
    for r in elt.itertuples():
        ev_mean[int(r.EventId)] = ev_mean.get(int(r.EventId), 0.0) + r.MeanLoss

BAND_EDGES = {}
for pg, sub in catalogue.groupby("peril_group"):
    v = np.array([ev_mean[int(e)] for e in sub.event_id])
    inner = np.geomspace(v.min(), v.max(), N_BANDS - 1)
    BAND_EDGES[pg] = np.concatenate([[-np.inf], inner, [np.inf]])

for pg, e in BAND_EDGES.items():
    print(f"{pg:9s} {N_BANDS} bands, inner edges "
          f"{np.array2string(e[1:-1], precision=1, separator=', ')}")
print("derived from ELT MeanLoss only - no simulated data, hence no barrier")

# --- R8: integer cell codes - peril_code*1000 + band ------------------------
# The hot paths (pass 2, reconstruction) group and merge on cells millions of
# times at production T; per-row f-strings are Python-level work the bottleneck
# hierarchy says to remove first. Labels are reconstructed only for display/audit.
PERIL_CODE = {pg: i for i, pg in enumerate(sorted(BAND_EDGES))}
CODE_PERIL = {i: pg for pg, i in PERIL_CODE.items()}
cell_label = lambda c: f"{CODE_PERIL[c // 1000]}|B{c % 1000}"
print(f"cell encoding: peril_code*1000 + band, peril codes {PERIL_CODE}")

## 3 · The generation worker (§10.2)

Written to a module file so it is importable by spawned processes. On Windows, `ProcessPoolExecutor`
uses *spawn* and cannot pickle functions defined in a notebook cell — a module file is the portable
fix, and it keeps one source of truth for serial and parallel paths.

`SeedSequence(ROOT_SEED).spawn(N_CHUNKS)` gives one independent stream per **chunk**, never
`seed + worker_id`. Because streams are keyed to the fixed chunk grid, the result is identical however
many workers consume them.

**R7 — the generator is vectorised across events.** Revision 1 looped per (event, account): one
Poisson call, one `beta.ppf` call and one mini-DataFrame per pair, so runtime scaled with *catalogue
size* rather than occurrence count (each scipy call carries ~0.1–0.15 ms of fixed dispatch overhead —
measured 12× slower at 4,000 events, and real ELTs run 10⁴–10⁵). Revision 2 draws counts as one
`(E, T)` Poisson call and severity as **one array-parameter `beta.ppf` call per account per chunk**.
The values are the same element-wise computation; the RNG *consumption layout* changes, so revision-2
books are statistically identical but not bit-identical to revision-1 books — which is why the
audit record now versions the generator and hashes the worker source (R2).

**R9 — shared state ships once per worker.** Tasks previously pickled `PARAMS`, `in_R`, the band
edges and the peril map per task — at demo scale this overhead is why 1 worker beat 4 in the §5
timings. Shared state now travels via `ProcessPoolExecutor(initializer=...)`, once per worker
process, and a task is just `(year_lo, year_hi, seed)`.

In [ ]:
WORKER_SRC = '''
import numpy as np, pandas as pd
from scipy import stats

NOMINAL_YR = np.datetime64("2000-01-01")
OCC_KEY    = ["year_id", "event_id", "loss_date"]
ACC_COLS   = ["accnt_no"] + OCC_KEY + ["loss"]

_G = {}

def init_shared(shared):
    # R9: runs once per worker process via ProcessPoolExecutor(initializer=...),
    # so shared state is pickled per WORKER, not per task.
    _G.clear()
    _G.update(shared)


def assign_cell(df, band_edges, peril_code):
    # Cell = peril group x band of the row's OWN loss (section 5.2).
    # R8: integer codes peril_code*1000 + band - no per-row string construction.
    pg   = df["peril_group"].to_numpy()
    loss = df["loss"].to_numpy()
    cell = np.zeros(len(df), dtype=np.int64)
    for g, e in band_edges.items():
        m = pg == g
        if m.any():
            cell[m] = peril_code[g] * 1000 + (np.searchsorted(e, loss[m], side="right") - 1)
    return cell


def generate_chunk(task):
    # Account-level rows for trial years [year_lo, year_hi). Deterministic given seed.
    #
    # R7: vectorised across events. Counts are one (E, T) Poisson draw; severity is ONE
    # array-parameter beta.ppf call per account per chunk, so cost scales with occurrence
    # count, not catalogue size. Memory is O(E x CHUNK_YEARS) for the count matrix: for
    # 1e4-1e5-event catalogues shrink CHUNK_YEARS (the recorded free parameter) or block
    # events. RNG consumption layout differs from revision 1: statistically identical,
    # NOT bit-identical - the audit's generator field versions this.
    year_lo, year_hi, seed = task
    params, cat, accounts = _G["params"], _G["cat"], _G["accounts"]
    rng, T = np.random.default_rng(seed), year_hi - year_lo
    ev_ids = np.array([e for e, _ in cat], dtype=np.int64)
    rates  = np.array([r for _, r in cat])
    E      = len(ev_ids)

    cnt = rng.poisson(rates[:, None], (E, T))            # occurrences per (event, year)
    assert cnt.max() <= 365, ("an event exceeded 365 occurrences in a year - the "
                              "consecutive-day scheme would collide on the occurrence key (R10)")
    c = cnt.ravel()                                      # event-major, year-minor
    M = int(c.sum())
    if M == 0:
        return pd.DataFrame(columns=ACC_COLS)

    m_ev   = cnt.sum(axis=1)                             # occurrences per event
    ev_row = np.repeat(np.arange(E), m_ev)               # event index per occurrence
    yrs    = np.repeat(np.tile(np.arange(T), E), c)      # local year per occurrence
    starts = np.concatenate([[0], np.cumsum(c)])[:-1]
    occ_i  = np.arange(M) - np.repeat(starts, c)         # index within (event, year)
    b_day  = np.repeat(rng.integers(1, 366, (E, T)).ravel(), c)
    day    = 1 + (b_day - 1 + occ_i) % 365
    dates  = NOMINAL_YR + (day - 1).astype("timedelta64[D]")
    z_corr = rng.standard_normal(M)                      # shared across accounts per occurrence

    pos  = {int(e): j for j, e in enumerate(ev_ids)}
    cols = {k: [] for k in ACC_COLS}
    for acct in accounts:
        pa = params[acct]
        a_e  = np.full(E, np.nan); b_e = np.full(E, np.nan); ex_e = np.full(E, np.nan)
        wc_e = np.full(E, np.nan); wi_e = np.full(E, np.nan)
        present = np.zeros(E, bool)
        for e, (a_, b_, ex_, wc_, wi_) in pa.items():
            j = pos.get(int(e))
            if j is not None:
                present[j] = True
                a_e[j], b_e[j], ex_e[j], wc_e[j], wi_e[j] = a_, b_, ex_, wc_, wi_
        sel = present[ev_row]
        k   = int(sel.sum())
        if k == 0:
            continue
        er   = ev_row[sel]
        u    = stats.norm.cdf(wc_e[er] * z_corr[sel] + wi_e[er] * rng.standard_normal(k))
        loss = stats.beta.ppf(u, a_e[er], b_e[er]) * ex_e[er]   # ONE ppf call per account (R7)
        cols["accnt_no"].append(np.full(k, acct))
        cols["event_id"].append(ev_ids[er])
        cols["year_id"].append(yrs[sel] + year_lo)
        cols["loss_date"].append(dates[sel])
        cols["loss"].append(loss)
    if not cols["loss"]:
        return pd.DataFrame(columns=ACC_COLS)
    return pd.DataFrame({k: np.concatenate(v) for k, v in cols.items()})[ACC_COLS]


def pass1_reduce(task):
    # PASS 1: portfolio rows + per-year Y (AEP) and X (OEP), and the envelope bounds
    # Low/Env at both grains (E1). All are per-year SUMS or MAXES, therefore reducible
    # under any partition; account rows are still discarded here, memory stays O(chunk).
    acct = generate_chunk(task)
    w_lo, w_hi = _G["w_lo"], _G["w_hi"]
    ls  = acct["loss"].to_numpy()
    occ = (acct.assign(lo_=acct["accnt_no"].map(w_lo).to_numpy() * ls,
                       hi_=acct["accnt_no"].map(w_hi).to_numpy() * ls)
               .groupby(OCC_KEY, as_index=False)[["loss", "lo_", "hi_"]].sum())
    del acct
    g = occ.groupby("year_id")
    return (occ[OCC_KEY + ["loss"]],
            g["loss"].sum(), g["loss"].max(),          # Y, X          (base book)
            g["lo_"].sum(),  g["hi_"].sum(),           # Low_Y, Env_Y  (AEP grain)
            g["lo_"].max(),  g["hi_"].max())           # Low_X, Env_X  (OEP grain)


def pass2_split(task):
    # PASS 2: retained tail rows + REDUCIBLE per-cell sums for the body. Never a share.
    in_R, band_edges = _G["in_R"], _G["band_edges"]
    peril_of, peril_code = _G["peril_of"], _G["peril_code"]
    acct = generate_chunk(task)
    mask = in_R[acct["year_id"].to_numpy()]
    tail, body = acct[mask], acct[~mask]
    del acct
    if len(body) == 0:
        empty = pd.Series(dtype=float)
        return tail, empty, empty, empty, pd.Series(dtype=int)
    port = body.groupby(OCC_KEY, as_index=False)["loss"].sum().rename(columns={"loss": "L_r"})
    port["peril_group"] = port["event_id"].map(peril_of)
    port["cell"]        = assign_cell(port.rename(columns={"L_r": "loss"}), band_edges, peril_code)
    ba  = body.merge(port[OCC_KEY + ["cell", "L_r"]], on=OCC_KEY, how="left")
    ba["s"] = ba["loss"] / ba["L_r"]
    g   = ba.groupby(["cell", "accnt_no"])
    # numerator, denominator, s-sum and row count: all SUMS, therefore reducible
    return (tail, g["loss"].sum(), port.groupby("cell")["L_r"].sum(),
            g["s"].sum(), port.groupby("cell").size())
'''

Path("cat_worker.py").write_text(WORKER_SRC)
WORKER_HASH = hashlib.sha256(WORKER_SRC.encode()).hexdigest()[:16]   # R2: part of book identity
sys.path.insert(0, str(Path.cwd()))
import cat_worker
importlib.reload(cat_worker)

def beta_ab(mu_, sd_):
    v = sd_ ** 2
    assert np.all(v < mu_ * (1 - mu_)), "infeasible Beta: SdIndep+SdCorr too large for MeanLoss"
    nu = mu_ * (1 - mu_) / v - 1
    return mu_ * nu, (1 - mu_) * nu

def elt_params(elt):
    p = {}
    for r in elt.itertuples():
        mdr, sdr = r.MeanLoss / r.Exposure, (r.SdIndep + r.SdCorr) / r.Exposure
        a, b     = beta_ab(mdr, sdr)
        w        = np.hypot(r.SdCorr, r.SdIndep)
        p[int(r.EventId)] = (a, b, r.Exposure, r.SdCorr / w, r.SdIndep / w)
    return p

PARAMS   = {a: elt_params(e) for a, e in ELTS.items()}
CAT_LIST = [(int(e), float(r)) for e, r in zip(catalogue.event_id, catalogue.Rate)]
BOUNDS   = list(range(0, T_YEARS + 1, CHUNK_YEARS))
SEEDS    = np.random.SeedSequence(ROOT_SEED).spawn(N_CHUNKS)
TASKS1   = [(BOUNDS[c], BOUNDS[c + 1], SEEDS[c]) for c in range(N_CHUNKS)]  # (lo, hi, seed) only (R9)
SHARED1  = {"params": PARAMS, "cat": CAT_LIST, "accounts": ACCOUNTS,
            "w_lo": {a: float(W_LO) for a in ACCOUNTS},        # E1: per-account bounds -
            "w_hi": {a: float(W_HI) for a in ACCOUNTS}}        # heterogeneous W drops in here

def run_tasks(fn, tasks, workers, shared):
    cat_worker.init_shared(shared)          # parent too: serial path and direct module calls
    if workers <= 1:
        return [fn(t) for t in tasks]
    with ProcessPoolExecutor(max_workers=workers, initializer=cat_worker.init_shared,
                             initargs=(shared,)) as ex:   # shipped once per worker (R9)
        return list(ex.map(fn, tasks))

print(f"worker module written, source hash {WORKER_HASH}; "
      f"{N_CHUNKS} tasks, one spawned stream each")

## 4 · Pass 1 — portfolio table and the per-year reductions (§4.1.1–4.1.2)

`Y_t` = Σ occurrences in year t (AEP basis), `X_t` = max occurrence in year t (OEP basis).
Account rows are discarded inside the worker; peak memory is O(chunk).

Accumulated with `np.add.at` / `np.maximum.at` rather than `Y[idx] = ...`. Assignment is only safe
because chunks happen to partition years — it clobbers silently if anyone ever chunks by event or by
replicate. The `.at` forms are correct under any partition and cost nothing.

**E1:** pass 1 additionally accumulates `Low_Y/Env_Y` (per-year weighted sums) and `Low_X/Env_X`
(per-year maxima of per-occurrence weighted sums) for the envelope bounds — same reduction rules,
same O(chunk) memory, and for the demo’s uniform box they must equal `W_LO·Y` and `W_HI·Y` exactly,
which is asserted as an identity check on the general per-account code path.

In [ ]:
t0 = time.time()
res1 = run_tasks(cat_worker.pass1_reduce, TASKS1, N_WORKERS, SHARED1)

Y     = np.zeros(T_YEARS); X     = np.zeros(T_YEARS)
LOW_Y = np.zeros(T_YEARS); ENV_Y = np.zeros(T_YEARS)      # E1: envelope accumulators -
LOW_X = np.zeros(T_YEARS); ENV_X = np.zeros(T_YEARS)      # O(T) memory, reducible
port_parts = []
for port, y_s, x_m, lo_s, hi_s, lo_m, hi_m in res1:
    iy = y_s.index.to_numpy()
    np.add.at(Y, iy, y_s.to_numpy())                      # safe under ANY partition
    np.add.at(LOW_Y, iy, lo_s.to_numpy())
    np.add.at(ENV_Y, iy, hi_s.to_numpy())
    ix = x_m.index.to_numpy()
    np.maximum.at(X, ix, x_m.to_numpy())
    np.maximum.at(LOW_X, ix, lo_m.to_numpy())
    np.maximum.at(ENV_X, ix, hi_m.to_numpy())
    port_parts.append(port)
portfolio_plt = pd.concat(port_parts, ignore_index=True)
del port_parts, res1

print(f"pass 1: {len(portfolio_plt):,} occurrences in {time.time()-t0:.1f}s on {N_WORKERS} workers")
display(portfolio_plt.head(4))
assert not portfolio_plt.duplicated(OCC_KEY).any(), "occurrence key is not unique"
assert np.isclose(Y.sum(), portfolio_plt["loss"].sum()), "Y accumulator lost mass"
# pointwise dominance must hold by construction; and because the demo box is UNIFORM,
# the envelopes must equal scalar multiples of Y and X exactly (identity check only -
# the code path is the general per-account one)
assert (LOW_Y <= Y + 1e-9).all() and (Y <= ENV_Y + 1e-9).all(), "AEP envelope violated"
assert (LOW_X <= X + 1e-9).all() and (X <= ENV_X + 1e-9).all(), "OEP envelope violated"
assert np.allclose(LOW_Y, W_LO * Y) and np.allclose(ENV_Y, W_HI * Y)
assert np.allclose(LOW_X, W_LO * X) and np.allclose(ENV_X, W_HI * X)
print("occurrence key unique; Y reconciles; envelope dominance + uniform-box identity OK")

## 5 · Determinism — worker count is not part of the run's identity

The document (§10.2) says worker count must be recorded because it forms part of the run's identity.
That is true only if the chunk grid is derived from it. With `CHUNK_YEARS` fixed, the same book comes
out on any number of cores — asserted here at 1, 2 and 4 workers.

In [ ]:
if not RUN_INVARIANCE_CHECKS:
    print("invariance rerun skipped (RUN_INVARIANCE_CHECKS = False); property established at demo T")
else:
    ref = None
    for w in [1, 2, 4]:
        t = time.time()
        out = run_tasks(cat_worker.pass1_reduce, TASKS1, w, SHARED1)
        acc = [np.zeros(T_YEARS) for _ in range(6)]
        for _, y_s, x_m, lo_s, hi_s, lo_m, hi_m in out:
            iy, ix = y_s.index.to_numpy(), x_m.index.to_numpy()
            np.add.at(acc[0], iy, y_s.to_numpy());     np.maximum.at(acc[1], ix, x_m.to_numpy())
            np.add.at(acc[2], iy, lo_s.to_numpy());    np.add.at(acc[3], iy, hi_s.to_numpy())
            np.maximum.at(acc[4], ix, lo_m.to_numpy()); np.maximum.at(acc[5], ix, hi_m.to_numpy())
        h = hashlib.sha256(np.concatenate(acc).tobytes()).hexdigest()[:16]
        print(f"  {w} worker(s): {time.time()-t:5.1f}s   Y|X|Low|Env digest {h}")
        assert (np.array_equal(acc[0], Y) and np.array_equal(acc[1], X)
                and np.array_equal(acc[2], LOW_Y) and np.array_equal(acc[3], ENV_Y)
                and np.array_equal(acc[4], LOW_X) and np.array_equal(acc[5], ENV_X)), \
            "worker count changed the book"
        ref = h if ref is None else ref
        assert h == ref
    print("bit-identical across worker counts OK  -> record chunk_years, not n_workers")

## 6 · Retention years — union of the AEP and OEP tails (§4.1.3)

`k = ⌈(1−q)T⌉`, `R = top-k{Y_t} ∪ top-k{X_t}`, rank-based rather than an interpolated percentile,
with deterministic tie-breaking on `year_id`.

The document's first point of care — *threshold on the year, not the row* — is checked by measuring
what a row-ranked cut would drop at two budgets.

**E2 — envelope retention.** Alongside the base union, retain `{t : Env(t) ≥ τ}` per grain with
`τ` the m-th largest `Low`. Proof of containment for every `v ∈ W`: `Low ≤ Y_v ≤ Env` pointwise
implies the m-th largest of `Y_v` is at least τ, so any α-tail year under `v` satisfies
`Env(t) ≥ Y_v(t) ≥ τ`; ties at τ are kept by `≥`, erring on retention. Uniform scalings never
re-rank, extending the guarantee to the cone `{c·v}`. The extra years beyond the base union are the
**measured price** of exact re-weighting — printed here, carried per-strategy in the appendix.

In [ ]:
k = int(np.ceil((1 - Q_RETAIN) * T_YEARS))

def top_k_years(metric, kk):
    return np.lexsort((np.arange(len(metric)), -metric))[:kk]     # ties -> lower year_id first

aep_tail, oep_tail = top_k_years(Y, k), top_k_years(X, k)
R_BASE = np.sort(np.union1d(aep_tail, oep_tail))

# --- E2: envelope retention -------------------------------------------------
# For any v in W:  Low(t) <= Y_v(t) <= Env(t) pointwise, so the m-th largest of Y_v is
# >= tau = m-th largest of Low. A year in the alpha-tail under v therefore satisfies
# Env(t) >= Y_v(t) >= (m-th largest of Y_v) >= tau. Retaining {t : Env(t) >= tau} thus
# contains the alpha-tail for EVERY v in W - and for every c*v with c > 0, because
# uniform scaling cannot re-rank years. Same argument at occurrence grain for OEP.
# Ties at tau are kept by >=, which errs on retention. One O(T) pass, no regeneration.
m_alloc = int(np.ceil((1 - ALPHA) * T_YEARS))
tau_y   = np.partition(LOW_Y, -m_alloc)[-m_alloc]
tau_x   = np.partition(LOW_X, -m_alloc)[-m_alloc]
R_ENV   = np.flatnonzero((ENV_Y >= tau_y) | (ENV_X >= tau_x))
R       = np.sort(np.union1d(R_BASE, R_ENV))
in_R    = np.zeros(T_YEARS, bool); in_R[R] = True

ovl = len(np.intersect1d(aep_tail, oep_tail))
print(f"k = {k:,} years per metric | m = {m_alloc:,} allocation-tail years")
print(f"AEP {len(aep_tail):,} | OEP {len(oep_tail):,} | base union {len(R_BASE):,} = "
      f"{len(R_BASE)/k:.2f}x single-metric | overlap {ovl:,} ({100*ovl/k:.0f}%)")
print(f"  document quotes 1.3-1.6x; measured {len(R_BASE)/k:.2f}x. It is a property of the book -")
print(f"  at {catalogue.Rate.sum():.1f} occurrences/yr single events dominate annual totals, so AEP")
print("  and OEP largely agree. Higher-frequency books separate them. Measure, do not assume.")
extra = len(np.setdiff1d(R_ENV, R_BASE))
print(f"envelope W = [{W_LO}, {W_HI}]: tau_Y = {tau_y:,.1f}, tau_X = {tau_x:,.1f} -> "
      f"{len(R_ENV):,} envelope years, {extra:,} beyond the base union")
print(f"retained |R| = {len(R):,} ({100*len(R)/T_YEARS:.2f}% of years); the envelope's price is "
      f"{extra:,} years = {100*extra/T_YEARS:.2f}pp - the measured cost of exact re-weighting on W")

alloc_years = top_k_years(Y, m_alloc)
assert in_R[alloc_years].all(), "allocation tail is not contained in the retained set"
print(f"allocation needs top {m_alloc:,} years by Y - all inside R OK")

row_order = np.argsort(-portfolio_plt["loss"].to_numpy(), kind="stable")
row_yrs   = portfolio_plt["year_id"].to_numpy()
for label, n_keep in [("same k as years", k),
                      ("same fraction of rows", int(np.ceil((1 - Q_RETAIN) * len(portfolio_plt))))]:
    kept   = np.unique(row_yrs[row_order[:n_keep]])
    missed = np.setdiff1d(alloc_years, kept)
    print(f"row-ranked cut, {label:24s}: {n_keep:6,} rows -> {len(kept):5,} years, misses "
          f"{len(missed):3,}/{m_alloc:,} allocation-tail years ({100*len(missed)/m_alloc:.1f}%)")
print("  years built from several moderate occurrences are what a row rank loses. Rank years.")

## 7 · Pass 2 — retain the tail, reduce the body (§4.1.4, §4.1.5, §5.1, §5.3)

One pass does both jobs. **Retention is decided first and the body defined as the residual**, so a row
in a retained year never receives a band and never enters a share calculation — no double counting,
and the shares are calibrated on exactly the population they are applied to (§5.3).

Workers return **sums**, never shares: `Σ Lₐ,ᵣ` and `Σ Lᵣ` per cell. The division happens once, after
the reduction. Averaging per-worker shares would reintroduce the §5.1 unweighted-estimator bias keyed
on which worker saw which occurrences — the same covariance term, at worker granularity.

In [ ]:
SHARED2 = {**SHARED1, "in_R": in_R, "band_edges": BAND_EDGES,
           "peril_of": PERIL_OF, "peril_code": PERIL_CODE}
t0 = time.time()
res2 = run_tasks(cat_worker.pass2_split, TASKS1, N_WORKERS, SHARED2)

tail_rows = pd.concat([r[0] for r in res2], ignore_index=True)

def reduce_(parts, lv):
    parts = [p for p in parts if len(p)]                 # R10: all-empty guard
    return pd.concat(parts).groupby(level=lv).sum() if parts else pd.Series(dtype=float)

NUM   = reduce_([r[1] for r in res2], [0, 1])      # sum_r L_{a,r}  per (cell, account)
DEN   = reduce_([r[2] for r in res2], 0)           # sum_r L_r      per cell
S_SUM = reduce_([r[3] for r in res2], [0, 1])      # sum_r s_{a,r}  per (cell, account)
N_C   = reduce_([r[4] for r in res2], 0)           # n              per cell
del res2

# --- the single division, AFTER the reduction (section 5.1) ---------------
share_matrix = (NUM / DEN).rename("share").reset_index() \
                 .pivot(index="cell", columns="accnt_no", values="share").fillna(0.0)

print(f"pass 2: {len(tail_rows):,} retained account rows in {time.time()-t0:.1f}s")
print(f"retained {len(R):,}/{T_YEARS:,} years ({100*len(R)/T_YEARS:.2f}%), "
      f"{len(share_matrix)} cells")
assert np.allclose(share_matrix.sum(axis=1), 1.0), "shares within a cell must sum to 1"
print(f"every cell's shares sum to 1 OK   sparsity "
      f"{100*(share_matrix==0).to_numpy().mean():.0f}% zeros")
display(share_matrix.rename(index=cell_label).head(6))

In [ ]:
# --- the reduction must be worker-count invariant, and must NOT be an average
if not RUN_INVARIANCE_CHECKS:
    print("invariance rerun skipped (RUN_INVARIANCE_CHECKS = False)")
else:
    for w in [1, 4]:
        out = run_tasks(cat_worker.pass2_split, TASKS1, w, SHARED2)
        sm  = ((reduce_([r[1] for r in out], [0, 1]) / reduce_([r[2] for r in out], 0))
               .rename("share").reset_index()
               .pivot(index="cell", columns="accnt_no", values="share").fillna(0.0))
        # R10: two-sided comparison - reindex_like would silently drop EXTRA cells in sm
        assert sm.index.equals(share_matrix.index) and sm.columns.equals(share_matrix.columns), \
            "cell/account sets differ across worker counts"
        assert np.allclose(sm.to_numpy(), share_matrix.to_numpy()), \
            "share reduction is not worker-count invariant"
        if w == 4:
            # what averaging per-worker shares would have done instead
            # (the flat cell-indexed DEN broadcasts across the MultiIndex cell level)
            per_worker = [ (r[1] / r[2]).unstack(fill_value=0.0) for r in out if len(r[1]) ]
            avg = sum(p.reindex_like(share_matrix).fillna(0.0) for p in per_worker) / len(per_worker)
            gap = (avg - share_matrix).abs().to_numpy().max()
            print(f"averaging per-worker shares instead would shift a share by up to {gap:.2e} "
                  f"({100*gap/share_matrix.to_numpy().max():.2f}% of the largest share)")
        del out
    print("sum-then-divide reduction is worker-count invariant OK")

## 8 · The validation run (§8) — *not* the production path

The protocol needs the full account table under common random numbers, so it is regenerated here in
full. This is the only place it is materialised; both production passes above stayed O(chunk).

Regeneration from the same spawned seeds is asserted bit-identical against the retained rows from
pass 2 — without that, approximation error cannot be distinguished from sampling noise.

In [ ]:
account_plt_TRUE = pd.concat(run_tasks(cat_worker.generate_chunk, TASKS1, N_WORKERS, SHARED1),
                             ignore_index=True)
FULL_ACCOUNT_ROWS = len(account_plt_TRUE)

lhs = tail_rows.sort_values(ACC_COLS).reset_index(drop=True)
rhs = (account_plt_TRUE[in_R[account_plt_TRUE["year_id"].to_numpy()]]
       .sort_values(ACC_COLS).reset_index(drop=True))
pd.testing.assert_frame_equal(lhs, rhs, check_exact=True)   # R4: every column, exactly
print(f"pass 2 reproduces the validation run bit-for-bit over {len(lhs):,} rows OK")

aal_sim  = account_plt_TRUE.groupby("accnt_no")["loss"].sum() / T_YEARS
var_true = pd.Series({a: (e.Rate * (e.MeanLoss ** 2 + (e.SdIndep + e.SdCorr) ** 2)).sum()
                      for a, e in ELTS.items()})
se_aal   = np.sqrt(var_true / T_YEARS)
chk = pd.DataFrame({"AAL analytic": aal_true, "AAL simulated": aal_sim, "SE": se_aal,
                    "z": (aal_sim - aal_true) / se_aal}).loc[ACCOUNTS]
display(chk)
assert np.all(np.abs(chk["z"]) < 4.0), "AAL closure failed beyond Monte Carlo error"

n_acc_per_event = FULL_ACCOUNT_ROWS / len(portfolio_plt)
print(f"account table {FULL_ACCOUNT_ROWS:,} rows | mean accounts per occurrence "
      f"{n_acc_per_event:.2f} (the term section 9 says to measure before committing)")

## 9 · Why loss-weighted — the covariance diagnostic (§5.1)

For the unweighted estimator `s̃ₐ = (1/n) Σ sₐ,ᵣ` the per-cell reconstruction error is exactly
`Σᵣ Lₐ,ᵣ − s̃ₐ Σᵣ Lᵣ = n · Covᵣ(sₐ,ᵣ, Lᵣ)`.

**Correction 2:** that is the *population* covariance (÷n). `np.cov` defaults to `ddof=1`, which breaks
the identity by `n/(n−1)`. Confirmed by brute force on densified vectors below.

The sign of the bias is the point: an account whose share rises with occurrence size is *understated*
by the unweighted estimator — precisely the concentrated account a capital allocation must not
under-charge.

In [ ]:
diag = (pd.DataFrame({"L_a": NUM, "s_sum": S_SUM}).reset_index()
          .merge(DEN.rename("L_tot").reset_index(), on="cell")
          .merge(N_C.rename("n").reset_index(), on="cell"))
diag["share_weighted"]   = diag["L_a"] / diag["L_tot"]
diag["share_unweighted"] = diag["s_sum"] / diag["n"]        # s = 0 where absent -> divide by full n
diag["err_unweighted"]   = diag["L_a"] - diag["share_unweighted"] * diag["L_tot"]

body_acct = (account_plt_TRUE[~in_R[account_plt_TRUE["year_id"].to_numpy()]]).copy()
body_port = portfolio_plt[~in_R[portfolio_plt["year_id"].to_numpy()]].copy()
body_port["peril_group"] = body_port["event_id"].map(PERIL_OF)
body_port["cell"] = cat_worker.assign_cell(body_port, BAND_EDGES, PERIL_CODE)
body_acct = body_acct.merge(body_port[OCC_KEY + ["cell", "loss"]]
                            .rename(columns={"loss": "L_r"}), on=OCC_KEY, how="left")
body_acct["s"] = body_acct["loss"] / body_acct["L_r"]

rows = []
for cell_, acct_ in diag.nlargest(3, "L_a")[["cell", "accnt_no"]].itertuples(index=False):
    prt = body_port.loc[body_port["cell"] == cell_, OCC_KEY + ["loss"]]
    sa  = body_acct.loc[(body_acct["cell"] == cell_) & (body_acct["accnt_no"] == acct_),
                        OCC_KEY + ["s"]]
    d   = prt.merge(sa, on=OCC_KEY, how="left").fillna({"s": 0.0})      # densify: absent -> 0
    n_  = len(d)
    rows.append({"cell": cell_label(cell_), "accnt_no": acct_, "n": n_,
                 "closed form": float(diag.loc[(diag.cell == cell_) &
                                               (diag.accnt_no == acct_), "err_unweighted"].iloc[0]),
                 "n*cov ddof=0": n_ * np.cov(d["s"], d["loss"], ddof=0)[0, 1],
                 "n*cov ddof=1": n_ * np.cov(d["s"], d["loss"], ddof=1)[0, 1]})
chkc = pd.DataFrame(rows)
chkc["ddof=1 error %"] = 100 * (chkc["n*cov ddof=1"] / chkc["closed form"] - 1)
display(chkc)
assert np.allclose(chkc["closed form"], chkc["n*cov ddof=0"], rtol=1e-9)
assert not np.allclose(chkc["closed form"], chkc["n*cov ddof=1"], rtol=1e-9)
print("identity holds against the POPULATION covariance OK  - np.cov's ddof=1 misses it (CORRECTION 2)")

In [ ]:
LADDER = {a: max(hi / lo for n_, p_, lo, hi, *_ in SPEC if n_ == a) for a in ACCOUNTS}
fl   = [a for a in ACCOUNTS if a.startswith("FL")]
# R10: select the columns before apply - version-agnostic (no include_groups, no
# grouping-column warnings on any pandas from 1.5 through 3.x)
bias = (diag[diag.accnt_no.isin(fl)]
        .groupby("accnt_no")[["share_unweighted", "share_weighted", "n"]]
        .apply(lambda d: np.average(d["share_unweighted"] / d["share_weighted"] - 1,
                                    weights=d["n"]))
        .rename("unweighted/weighted - 1"))
summary = pd.concat([bias, pd.Series(LADDER, name="ladder steepness hi/lo").loc[fl]], axis=1)
display(summary.sort_values("ladder steepness hi/lo", ascending=False))
steep, flat = summary["ladder steepness hi/lo"].idxmax(), summary["ladder steepness hi/lo"].idxmin()
print(f"steepest ({steep}) biased {summary.loc[steep,'unweighted/weighted - 1']:+.2%}, "
      f"flattest ({flat}) biased {summary.loc[flat,'unweighted/weighted - 1']:+.2%}")
# R5: the ORDERING is a population prediction (5.1), not a finite-sample identity - warn, don't assert
if summary.loc[steep, "unweighted/weighted - 1"] < summary.loc[flat, "unweighted/weighted - 1"]:
    print("  unweighted understates the concentrated account, as section 5.1 predicts")
else:
    print("  WARNING: bias ordering not resolved on this book/sample size - "
          "the sign is 5.1's population prediction; statistical, not an identity (R5)")

worst = (diag.assign(blur=diag["err_unweighted"].abs()).groupby("cell")["blur"].sum()
             .sort_values(ascending=False).head(5))
print()
print("cells blurring the most share variation (section 8 test 3 targets):")
display(worst.rename(index=cell_label))

## 10 · Persist, record provenance, discard (§4.1.6, §10.3)

`n_workers` is deliberately **absent** from the audit record — `chunk_years` is what fixes the book.

Also: the document's "partitioned by year" is an anti-pattern at this scale — `(1−q)T` retained years
gives ~1,200 directories of ~25 rows each, the small-file problem. A single file sorted on `year_id`
with bounded row groups gives the same predicate pushdown from Parquet row-group statistics far more
cheaply. Parallel writers should emit one part file each, which is the legitimate reason for multiple
files.

**R1/R2:** the severity hash is content-addressed (account label included, order-independent), and
the audit additionally pins the generator version, the worker-source hash and the library versions —
without those, "same seed" does not imply "same book".

In [ ]:
# R1: content-addressed severity hash - account label included, construction-order
# independent. The old hash concatenated dict values only: swapping which account owns
# which ELT while preserving insertion order COLLIDED, and reordering SPEC rows for an
# identical book false-alarmed.
severity_hash = hashlib.sha256(
    pd.concat([e.assign(accnt_no=a).sort_values("EventId")
               for a, e in sorted(ELTS.items())], ignore_index=True)
      .round(10).to_csv(index=False).encode()).hexdigest()[:16]

portfolio_plt.to_parquet(STORE / "portfolio_plt.parquet", index=False, compression="zstd")
share_matrix.to_parquet(STORE / "share_matrix.parquet", compression="zstd")
tail_rows = tail_rows.sort_values(["year_id"] + OCC_KEY[1:] + ["accnt_no"]).reset_index(drop=True)
tail_rows.to_parquet(STORE / "tail" / "part-00000.parquet", index=False,
                     compression="zstd", row_group_size=50_000)

# R2: CRN bit-reproducibility is only DEFINED at fixed library versions and a fixed RNG
# consumption order - beta.ppf implementations change across scipy releases, and any edit
# to the worker source changes the stream layout. Both now sit inside the audit record.
audit = {"root_seed": ROOT_SEED, "chunk_years": CHUNK_YEARS, "n_chunks": N_CHUNKS,
         "generator": "v2-vectorised (one ppf per account per chunk)",
         "worker_source_sha256": WORKER_HASH,
         "library_versions": {"python": platform.python_version(),
                              "numpy": np.__version__, "pandas": pd.__version__,
                              "scipy": scipy.__version__},
         "platform": platform.platform(),
         "severity_hash": severity_hash, "retention_percentile_q": Q_RETAIN,
         "retention_basis": "AEP union OEP", "n_retained_years": int(len(R)),
         "trial_years": T_YEARS, "alpha_supported_min": Q_RETAIN,
         "band_definitions": {p: [float(x) for x in e] for p, e in BAND_EDGES.items()},
         "band_basis": "a priori, log-spaced over ELT portfolio MeanLoss",
         "cell_encoding": {"formula": "peril_code*1000 + band", "peril_code": PERIL_CODE},
         "peril_grouping": {str(kk): v for kk, v in PERIL_OF.items()},
         "base_weight_vector": {a: 1.0 for a in ACCOUNTS},
         "weight_envelope": {"w_lo": {a: float(W_LO) for a in ACCOUNTS},   # E3: the guard
                             "w_hi": {a: float(W_HI) for a in ACCOUNTS},   # reads these
                             "alpha_envelope": ALPHA},
         "catlib_version": "1.0.0"}
(STORE / "audit.json").write_text(json.dumps(audit, indent=2))

du = lambda p: (sum(f.stat().st_size for f in Path(p).rglob("*") if f.is_file())
                if Path(p).is_dir() else Path(p).stat().st_size)
sizes = pd.Series({f.name: du(f) / 1e6 for f in sorted(STORE.iterdir())}, name="MB")
display(sizes.to_frame())
print(f"total store {sizes.sum():.2f} MB  |  account table would have been "
      f"{FULL_ACCOUNT_ROWS:,} rows (discarded in production)")

## 11 · Reconstruction and allocation (§4.2)

    L̂ₐ,ᵣ = Lₐ,ᵣ read from the tail store,  if year(r) ∈ R
         = s̄ₐ,c(r) · Lᵣ                    otherwise

Because `α ≥ q` the allocation reads stored data only — **the reconstruction is never exercised on the
allocation path**. It is built and tested anyway, because everything downstream of the allocation
does exercise it.

**E3 — the re-weighted read.** For `w` inside the contracted cone, every α-tail year under `w` is
retained (E2), so ranking the *retained* years by their exact re-weighted annual sums
`Σₐ wₐ Aₐ,ₜ` recovers the true tail, and `co-TVaRₐ(w) = E[wₐ Aₐ | tail]` is exact from the tail
store alone — no reconstruction on the allocation path, exactly as at `w = 1`. Uniform weights take
the base path and scale. Everything the readers need is the per-year×account matrix `A_TAIL`.

In [ ]:
# --- E3: which weightings does the store support? ---------------------------
W_LO_S = pd.Series(audit["weight_envelope"]["w_lo"]).reindex(ACCOUNTS)
W_HI_S = pd.Series(audit["weight_envelope"]["w_hi"]).reindex(ACCOUNTS)
BASE_W = pd.Series(audit["base_weight_vector"]).reindex(ACCOUNTS)

def assert_weights_supported(w, tol=1e-9):
    """Classify a weight vector against the contracted envelope (read from the audit).

    uniform          -> ranking-invariant: any alpha >= q, result scales by c
    cone {c*v, v in W} -> exact at alpha >= ALPHA via envelope retention (E2)
    otherwise        -> blocked: body rows cannot re-rank (7.1); regenerate or widen W
    The cone test: exists c > 0 with w/c in W  <=>  max_a(w_a/hi_a) <= min_a(w_a/lo_a).
    """
    w_ = pd.Series(w).reindex(BASE_W.index).fillna(1.0)
    assert (w_ > 0).all(), "weights must be positive"
    ratio = (w_ / BASE_W).to_numpy()
    if np.allclose(ratio, ratio[0], rtol=tol):
        return "uniform", float(ratio[0])
    c_lo, c_hi = float((w_ / W_HI_S).max()), float((w_ / W_LO_S).min())
    if c_lo <= c_hi * (1 + tol):
        return "cone", None
    raise ValueError(
        "weights outside the contracted envelope: no c > 0 puts w/c inside "
        f"W = [{W_LO_S.min():.2f}, {W_HI_S.max():.2f}] per account "
        f"(needed c in [{c_lo:.3f}, {c_hi:.3f}], empty). Body rows cannot re-rank (7.1): "
        "a fresh generation run is required, or widen W and re-run passes 1-2.")

# per-year x account annual sums over retained years: the ONLY object the readers need
A_TAIL = (tail_rows.groupby(["year_id", "accnt_no"])["loss"].sum().unstack(fill_value=0.0)
          .reindex(columns=ACCOUNTS).fillna(0.0))
YRS_R  = A_TAIL.index.to_numpy()

def co_tvar_from_store(alpha=ALPHA, w=None):
    m = int(np.ceil((1 - alpha) * T_YEARS))
    if w is not None:
        mode, c = assert_weights_supported(w)
        if mode == "uniform":
            alloc, tv = co_tvar_from_store(alpha)          # base ranking; asserts alpha >= q
            return alloc * c, tv * c
        # cone: every alpha-tail year under w is retained (E2), so ranking WITHIN the
        # retained set by the exact re-weighted annual sums recovers the true tail
        assert alpha >= ALPHA, (f"non-uniform weights are contracted at alpha >= {ALPHA} "
                                f"(the envelope level); got alpha = {alpha}")
        wv    = pd.Series(w).reindex(ACCOUNTS).to_numpy()
        Yw_R  = A_TAIL.to_numpy() @ wv
        sel   = np.lexsort((YRS_R, -Yw_R))[:m]             # ties -> lower year_id first
        per   = A_TAIL.to_numpy()[sel] * wv
        alloc = pd.Series(per.mean(axis=0), index=ACCOUNTS)
        return alloc, float(Yw_R[sel].mean())
    assert alpha >= Q_RETAIN, f"alpha={alpha} below retention q={Q_RETAIN}: outside the exact region"
    tail_y = top_k_years(Y, m)                             # strict exceedance: the m worst years
    per_yr = A_TAIL.reindex(index=tail_y).fillna(0.0)
    return per_yr.mean(axis=0), Y[tail_y].mean()

alloc_store, TVaR_store = co_tvar_from_store()
print(f"TVaR_{ALPHA} = {TVaR_store:,.2f}  (tail = {int(np.ceil((1-ALPHA)*T_YEARS)):,} years)")
display(alloc_store.rename("co-TVaR (from store)").to_frame())
assert abs(alloc_store.sum() - TVaR_store) < 1e-9 * TVaR_store
print(f"full allocation: sum {alloc_store.sum():,.6f} vs TVaR {TVaR_store:,.6f} OK")

def reconstruct_body(port_rows, edges=None, sm=None):
    df = port_rows.copy()
    df["peril_group"] = df["event_id"].map(PERIL_OF)
    df["cell"] = cat_worker.assign_cell(df, BAND_EDGES if edges is None else edges, PERIL_CODE)
    sm = share_matrix if sm is None else sm
    hit = sm.index.get_indexer(df["cell"])
    if (hit < 0).any():
        # R3: a cell with no body observations has no row in the share matrix; reindex
        # produced silent NaN allocated losses. Fail loudly instead.
        missing = sorted(set(df["cell"].to_numpy()[hit < 0]))
        raise KeyError("cells with no body observations: "
                       f"{[cell_label(c) for c in missing]} - the share matrix cannot "
                       "reconstruct them; refit the bands or coarsen the cell definition")
    return pd.DataFrame(sm.to_numpy()[hit] * df["loss"].to_numpy()[:, None],
                        columns=sm.columns, index=df.index)

## 12 · Validation protocol (§8) — with corrections 3 and 4

| test | measures | expected |
|---|---|---|
| 1 · per-account AAL | **implementation** of the share scheme | exact to machine precision |
| 2 · per-account co-TVaR | **retention step** | exact — tail stored verbatim |
| 3 · weighted vs unweighted | share variation blurred per cell | non-zero; targets refinement |
| 4 · per-body-row error *(added)* | **band adequacy** | the only test that grades resolution |
| — · additivity | the allocation identity, nothing else | passes regardless (§6.1) |

Tests 1 and 2 are pass/fail on *implementation*, not adequacy. A non-zero test 1 means unweighted
shares, or bands calibrated on the full table — never "bands too coarse".

In [ ]:
aal_truth = account_plt_TRUE.groupby("accnt_no")["loss"].sum().reindex(ACCOUNTS) / T_YEARS
tail_y    = top_k_years(Y, int(np.ceil((1 - ALPHA) * T_YEARS)))
tv_truth  = (account_plt_TRUE[np.isin(account_plt_TRUE["year_id"], tail_y)]
             .groupby(["year_id", "accnt_no"])["loss"].sum().unstack(fill_value=0.0)
             .reindex(index=tail_y, columns=ACCOUNTS).fillna(0.0).mean(axis=0))

body_recon = reconstruct_body(body_port)
aal_recon  = (tail_rows.groupby("accnt_no")["loss"].sum().reindex(ACCOUNTS).fillna(0.0)
              + body_recon.sum(axis=0).reindex(ACCOUNTS).fillna(0.0)) / T_YEARS

res = pd.DataFrame({"AAL truth": aal_truth, "AAL recon": aal_recon,
                    "AAL abs err": (aal_recon - aal_truth).abs(),
                    "co-TVaR truth": tv_truth, "co-TVaR store": alloc_store,
                    "co-TVaR abs err": (alloc_store - tv_truth).abs()})
display(res)
t1, t2 = res["AAL abs err"].max(), res["co-TVaR abs err"].max()
print(f"TEST 1  per-account AAL     max abs err {t1:.3e}  (expect ~0: exact by construction)")
print(f"TEST 2  per-account co-TVaR max abs err {t2:.3e}  (expect ~0: tail stored verbatim)")
assert t1 < 1e-9 * aal_truth.max(), "AAL mismatch -> implementation defect, not band resolution"
assert t2 < 1e-9 * tv_truth.max(),  "co-TVaR mismatch -> defect in the retention step"

body_truth = (body_acct.pivot_table(index=OCC_KEY, columns="accnt_no", values="loss",
                                    aggfunc="sum", fill_value=0.0)
              .reindex(columns=ACCOUNTS).fillna(0.0))
body_hat = (body_recon.set_index(pd.MultiIndex.from_frame(body_port[OCC_KEY]))
            .reindex(body_truth.index).reindex(columns=ACCOUNTS).fillna(0.0))
denom = body_truth.to_numpy().sum(axis=1)
rel   = np.abs(body_hat.to_numpy() - body_truth.to_numpy()).sum(axis=1) / np.where(denom > 0, denom, np.nan)
print()
print("TEST 4  per-body-row reconstruction error (share of row loss misplaced):")
print(f"        median {np.nanmedian(rel):.1%} | p90 {np.nanpercentile(rel,90):.1%} | "
      f"p99 {np.nanpercentile(rel,99):.1%}")
print("        this grades band resolution - tests 1 and 2 cannot.")

In [ ]:
# --- ADVERSARIAL PROBE: collapse to one band per peril --------------------
EDGES_1 = {p: np.array([-np.inf, np.inf]) for p in BAND_EDGES}
bp_d = body_port.copy(); bp_d["cell"] = cat_worker.assign_cell(bp_d, EDGES_1, PERIL_CODE)
ba_d = body_acct.drop(columns=["cell"]).merge(bp_d[OCC_KEY + ["cell"]], on=OCC_KEY, how="left")
sm_d = ((ba_d.groupby(["cell", "accnt_no"])["loss"].sum() / bp_d.groupby("cell")["loss"].sum())
        .rename("share").reset_index().pivot(index="cell", columns="accnt_no",
                                             values="share").fillna(0.0))
rc_d  = reconstruct_body(body_port, EDGES_1, sm_d)
aal_d = (tail_rows.groupby("accnt_no")["loss"].sum().reindex(ACCOUNTS).fillna(0.0)
         + rc_d.sum(axis=0).reindex(ACCOUNTS).fillna(0.0)) / T_YEARS
hat_d = (rc_d.set_index(pd.MultiIndex.from_frame(body_port[OCC_KEY]))
         .reindex(body_truth.index).reindex(columns=ACCOUNTS).fillna(0.0))
rel_d = np.abs(hat_d.to_numpy() - body_truth.to_numpy()).sum(axis=1) / np.where(denom > 0, denom, np.nan)

print(f"{'metric':36s}{N_BANDS} bands/peril   1 band/peril")
print(f"{'TEST 1 per-account AAL max err':36s}{t1:13.3e}{(aal_d-aal_truth).abs().max():15.3e}")
print(f"{'TEST 2 per-account co-TVaR max err':36s}{t2:13.3e}{t2:15.3e}")
print(f"{'additivity sum(a) - TVaR':36s}{abs(alloc_store.sum()-TVaR_store):13.3e}"
      f"{abs(alloc_store.sum()-TVaR_store):15.3e}")
print(f"{'TEST 4 median body-row error':36s}{np.nanmedian(rel):12.1%}{np.nanmedian(rel_d):15.1%}")
assert (aal_d - aal_truth).abs().max() < 1e-9 * aal_truth.max(), \
    "AAL is exact for ANY partition - aggregate-then-divide guarantees it"
print()
print("AAL, co-TVaR and additivity all pass exactly under a deliberately useless compression.")
print("Only test 4 sees it. That is corrections 3 and 4 in one table.")

## 13 · From binding limitation to contracted capability (§7.1 + E1–E4)

Within a cell every body row is a scaled copy of one account profile, so a weight change `w` scales
the whole cell by the *same* multiplier `Σₐ wₐ s̄ₐ,c`: within-cell year ordering is frozen at
whatever the base book produced. The illustration below reproduces the document’s example — with
**correction 5** to its arithmetic — and it is the reason body reconstruction can *never* substitute
for retention when weights change.

Revision 2 answered that limitation with a prohibition: block every non-uniform re-weighting.
Revision 3 replaces the prohibition with a **contract**. The perturbation family is declared up
front — per-account bounds `W = ∏[w_lo,ₐ, w_hi,ₐ]` — and pass 1 streams two extra reducible
accumulators per grain, `Low(t) = Σₐ w_lo,ₐ Lₐ,ₜ` and `Env(t) = Σₐ w_hi,ₐ Lₐ,ₜ`. Retaining every
year with `Env(t) ≥ τ` (τ = m-th largest `Low`) contains the α-tail for **every** `v ∈ W` — the
one-line proof sits in §6 — and hence for the whole cone `{c·v : c > 0}`, since uniform scaling
never re-ranks. Inside the cone, re-weighted co-TVaR is **exact and read from the tail store
alone**; the supported-set test is a spread condition, `maxₐ(wₐ/hiₐ) ≤ minₐ(wₐ/loₐ)`. Outside it,
the guard blocks with an actionable message instead of a silent wrong answer. The retention margin
now has a price *and* a meaning: the extra years measured in §6 are what exactness over `W` costs.

The envelope is the **locality principle in storage form**. The marginal-impact methodology’s
organizing rule is that a marginal is a directional derivative at today’s mix — never a budget for
a large move. `W = [0.85, 1.20]` exact, `3×` blocked with “regenerate”, is that rule made
operational: the compression scheme and the allocation methodology now agree structurally rather
than by coincidence.

In [ ]:
ill = pd.DataFrame({"A": [8, 1, 2], "B": [1, 10, 2], "C": [1, 1, 11]}, index=[1, 2, 3])
ill["Total"] = ill.sum(axis=1)
s_bar = ill[["A", "B", "C"]].sum() / ill["Total"].sum()
print("cell shares:", {kk: f"{v:.4f} ({int(round(v*37))}/37)" for kk, v in s_bar.items()})

w         = pd.Series({"A": 3.0, "B": 1.0, "C": 1.0})
truth_new = (ill[["A", "B", "C"]] * w).sum(axis=1)
mult      = float((w * s_bar).sum())
recon_new = ill["Total"] * mult
display(pd.DataFrame({"base total": ill["Total"], "truth after 3x A": truth_new,
                      "reconstruction": recon_new}))
print(f"uniform multiplier = {mult:.4f} for every row in the cell")
print("truth ordering          :", " > ".join(map(str, truth_new.sort_values(ascending=False).index)))
print("reconstruction ordering :", " > ".join(map(str, recon_new.sort_values(ascending=False).index)))
assert np.isclose(s_bar.sum(), 1.0)
assert np.allclose(s_bar.values, [11/37, 13/37, 13/37])
assert np.allclose(recon_new.values, [15.9459, 19.1351, 23.9189], atol=1e-3)
assert list(truth_new.sort_values(ascending=False).index) == [1, 3, 2]
assert list(recon_new.sort_values(ascending=False).index) == [3, 2, 1]
print("year 1 genuinely became the worst year; the reconstruction cannot surface it OK")
print()
print("CORRECTION 5 - the document's arithmetic here is wrong:")
print("  it states s_C = 12/37, but C's column is 1+1+11 = 13, so s_C = 13/37.")
print("  the tell is 11/37 + 13/37 + 12/37 = 36/37, and shares must sum to 1.")
print(f"  multiplier is 59/37 = {59/37:.4f}, not 58/37 = {58/37:.4f}, and the rows are")
print("  15.95 / 19.14 / 23.92, not 15.7 / 18.8 / 23.5.")
print("  the CONCLUSION is untouched: both orderings are as stated, the limitation is real.")

In [ ]:
# --- the guard: three zones instead of two (E3) ----------------------------
mode, c = assert_weights_supported({a: 1.15 for a in ACCOUNTS})
print(f"uniform 1.15x growth        : {mode}, multiplier {c:.2f} - any alpha >= q")
mode, _ = assert_weights_supported({**{a: 1.0 for a in ACCOUNTS}, "FL-01": 1.15})
print(f"mix change inside W (1.15x) : {mode} - EXACT at alpha >= {ALPHA} from the tail store")
GUARD_BLOCKS = False
try:
    assert_weights_supported({**{a: 1.0 for a in ACCOUNTS}, "FL-01": 3.0})
except ValueError as e:
    GUARD_BLOCKS = True
    print("mix change outside W (3x)   :", str(e).split(". ")[0], "-> BLOCKED OK")

# --- E4: sampled-w exactness against the full account table -----------------
# The containment proof covers all of cone(W); this validates the IMPLEMENTATION on
# interior points, corners, and cone points (c*v), against truth computed from the
# full CRN account table. Same samples are reused by the harness (CRN across strategies).
A_FULL = (account_plt_TRUE.groupby(["year_id", "accnt_no"])["loss"].sum().unstack(fill_value=0.0)
          .reindex(index=np.arange(T_YEARS), columns=ACCOUNTS).fillna(0.0))
rngw = np.random.default_rng(np.random.SeedSequence([ROOT_SEED, 777]))
W_SAMPLES  = [pd.Series(rngw.uniform(W_LO, W_HI, len(ACCOUNTS)), index=ACCOUNTS)
              for _ in range(6)]                                            # interior
W_SAMPLES += [pd.Series(np.where(rngw.random(len(ACCOUNTS)) < 0.5, W_LO, W_HI),
                        index=ACCOUNTS) for _ in range(4)]                  # corners
W_SAMPLES += [W_SAMPLES[0] * 1.8, W_SAMPLES[6] * 0.6]                       # cone points c*v

TAILS_W, TRUTH_W, env_rows = [], [], []
for i, w_ in enumerate(W_SAMPLES):
    wv    = w_.to_numpy()
    Yw    = A_FULL.to_numpy() @ wv
    tw    = top_k_years(Yw, m_alloc)
    truth = pd.Series((A_FULL.to_numpy()[tw] * wv).mean(axis=0), index=ACCOUNTS)
    TAILS_W.append(tw); TRUTH_W.append(truth)
    alloc, tv = co_tvar_from_store(ALPHA, w_)
    err     = float((alloc - truth).abs().max() / truth.abs().max())
    escapes = int(len(np.setdiff1d(tw, R_BASE)))            # would rev-2 retention have missed?
    assert bool(in_R[tw].all()), f"sample {i}: envelope containment violated"
    assert err < 1e-9, f"sample {i}: re-weighted allocation not exact (rel err {err:.2e})"
    assert abs(alloc.sum() - tv) < 1e-9 * tv, f"sample {i}: additivity broken"
    env_rows.append({"sample": i, "kind": "interior" if i < 6 else ("corner" if i < 10 else "cone"),
                     "spread": float(w_.max() / w_.min()),
                     "rel err vs truth": err, "tail years beyond base R": escapes})
env_df = pd.DataFrame(env_rows).set_index("sample")
display(env_df)
esc_max = int(env_df["tail years beyond base R"].max())
if esc_max > 0:
    print(f"up to {esc_max} of {m_alloc:,} true tail years per sample sit OUTSIDE the rev-2 base "
          "retention - the envelope is what makes those reads exact rather than silently wrong")
else:
    print("note: at this book/box the base margin happened to cover every sampled w (measured, "
          "book-dependent - R5). The envelope guarantee is what removes 'happened to'.")
ENV_CHECK = {"n": len(W_SAMPLES), "max_rel_err": float(env_df["rel err vs truth"].max()),
             "escapes_base_max": esc_max, "guard_blocks_outside": GUARD_BLOCKS}
print(f"E4: {ENV_CHECK['n']} sampled w in cone(W) - containment, exactness and additivity all hold")

# --- outside the contract, the limitation is unchanged: measure what 3x FL-01 does --
w_mix  = pd.Series({a: (3.0 if a == "FL-01" else 1.0) for a in ACCOUNTS})
Y_mix  = A_FULL.to_numpy() @ w_mix.reindex(ACCOUNTS).to_numpy()
R_mix  = top_k_years(Y_mix, m_alloc)
print()
print(f"after a true 3x on FL-01 (outside W): {100*in_R[R_mix].mean():.1f}% of the new "
      f"allocation-tail years happen to be retained, {len(np.setdiff1d(R_mix, R)):,} are not -")
print("that is why the guard stays STRICT: one missing year in 401 is exactly the invisible-error")
print("case 7.1 warns about, and 99.8% coverage is precisely the number that would tempt a")
print("tolerance. Near-total coverage is not a guarantee; the cone test is.")

## 14 · Storage sizing (§9) and the certification battery

**Post-review finding — the envelope changes the reduction order.** τ is set from ALPHA, not q, so
the envelope floor is invariant to §9’s reductions 1 (tighten q) and 2 (drop the union): with a
contracted `W` both collapse onto `R_ENV` exactly (measured in the harness as a set identity), and
**grain is the only remaining size lever**. The sizing formula’s retention/k factor (incl.
envelope) is where this enters the estimate; the W-width sweep in the harness prices the contract
itself.

In [ ]:
union_f  = len(R) / k
n_ev_yr  = catalogue.Rate.sum()
ev_in_R  = np.isin(portfolio_plt["year_id"], R).sum() / len(R)
N_T_doc  = (1 - Q_RETAIN) * T_YEARS * union_f * n_ev_yr * n_acc_per_event
N_T_fix  = (1 - Q_RETAIN) * T_YEARS * union_f * ev_in_R * n_acc_per_event

display(pd.Series({
    "trial years T": T_YEARS, "(1-q) tail years": (1 - Q_RETAIN) * T_YEARS,
    "retention/k factor (incl. envelope)": union_f, "events/yr (book average)": n_ev_yr,
    "events/yr (retained years)": ev_in_R, "accounts/event": n_acc_per_event,
    "N_T doc formula": N_T_doc, "N_T corrected": N_T_fix, "N_T actual": len(tail_rows),
    "full account rows": FULL_ACCOUNT_ROWS,
    "retained fraction %": 100 * len(tail_rows) / FULL_ACCOUNT_ROWS,
    "share matrix entries": share_matrix.size, "store MB": sizes.sum()}).to_frame("value"))

print("CORRECTION 6 - the section 9 formula uses book-average events/year, but retained years")
print("  are selected FOR annual severity, which correlates with occurrence count:")
print(f"  {ev_in_R:.2f} occurrences/yr inside R vs {n_ev_yr:.2f} book-wide ({ev_in_R/n_ev_yr:.2f}x),")
print(f"  so it under-predicts by {100*(1-N_T_doc/len(tail_rows)):.0f}%. Corrected: "
      f"{N_T_fix:,.0f} vs actual {len(tail_rows):,}.")
# R5: the corrected formula is a statistical prediction, not an identity - report, don't assert
gap = abs(N_T_fix / len(tail_rows) - 1)
print(f"  corrected sizing formula within {gap:.1%} of actual." if gap < 0.10 else
      f"  WARNING: corrected sizing formula off by {gap:.0%} on this book - "
      "statistical, book-dependent (R5)")
print(f"scaling to T = 1e6: ~{len(tail_rows)*1e6/T_YEARS/1e6:.1f}M retained rows vs "
      f"~{FULL_ACCOUNT_ROWS*1e6/T_YEARS/1e6:.0f}M full account rows")

In [ ]:
def certify_compression(alpha=ALPHA):
    rep = []
    add = lambda n, ok, d: rep.append((n, bool(ok), d))
    m_ = int(np.ceil((1 - alpha) * T_YEARS))

    add("alpha >= q (containment)", alpha >= Q_RETAIN,
        f"alpha={alpha}, q={Q_RETAIN} -> allocation tail inside retained set")
    add("retention keyed on YEAR", np.isin(top_k_years(Y, m_), R).all(),
        f"all {m_:,} allocation-tail years retained")
    add("no double counting", not np.isin(body_port["year_id"], R).any(),
        "no retained year appears in the body; one reconstruction source per row")
    add("bands a priori (no barrier)", True,
        f"{len(share_matrix)} cells, edges log-spaced over ELT MeanLoss")
    add("shares sum to 1 per cell", np.allclose(share_matrix.sum(axis=1), 1.0),
        f"max deviation {np.abs(share_matrix.sum(axis=1)-1).max():.2e}")
    a_, tv_ = co_tvar_from_store(alpha)
    add("full allocation (identity only)", abs(a_.sum() - tv_) < 1e-9 * tv_,
        f"gap {abs(a_.sum()-tv_):.2e} - NOT evidence of adequacy (6.1)")
    add("TEST 1 per-account AAL exact", t1 < 1e-9 * aal_truth.max(),
        f"max abs err {t1:.2e} (implementation test)")
    add("TEST 2 per-account co-TVaR exact", t2 < 1e-9 * tv_truth.max(),
        f"max abs err {t2:.2e} (retention test)")
    floor = np.nanmedian(rel_d)
    add("TEST 4 body error decomposed", np.nanmedian(rel) <= floor,
        f"median {np.nanmedian(rel):.1%} vs {floor:.1%} at 1 band -> only "
        f"{100*(floor-np.nanmedian(rel)):.1f}pp band-reducible (CORRECTION 7)")
    add("CRN reproducibility", True,
        f"pass 2 == validation run bit-for-bit; seed {ROOT_SEED}, {N_CHUNKS} streams")
    add("worker-count invariance", True,
        f"Y|X and share matrix identical at 1, 2, 4 workers; chunk_years={CHUNK_YEARS:,} fixes the book")
    add("shares reduced, not averaged", True,
        "workers return sums; the single division happens after the reduction")
    add("envelope containment (sampled W)", ENV_CHECK["n"] > 0 and ENV_CHECK["max_rel_err"] < 1e-9,
        f"{ENV_CHECK['n']} sampled w in cone(W): every alpha-tail inside R (E2/E4)")
    add("re-weighted co-TVaR exact on W", ENV_CHECK["max_rel_err"] < 1e-9,
        f"max rel err {ENV_CHECK['max_rel_err']:.2e} across sampled w; "
        f"up to {ENV_CHECK['escapes_base_max']} tail years/sample beyond base retention")
    add("outside-W guard armed", ENV_CHECK["guard_blocks_outside"],
        "weights outside cone(W) blocked with the widen-or-regenerate message (E3)")
    add("book identity pinned (R2)", True,
        f"seed + chunk_years + generator + worker source {WORKER_HASH} + library versions in audit")

    print(f"{'check':36s}{'status':8s}detail")
    for n, ok, d in rep:
        print(f"{n:36s}{'PASS' if ok else 'FAIL':8s}{d}")
    return rep

report = certify_compression()
assert all(ok for _, ok, _ in report)
print()
print("CORRECTION 7 - band resolution has far less leverage than 5.2 / test 3 imply.")
print(f"  {N_BANDS} bands/peril gives {np.nanmedian(rel):.1%} median body-row error; 1 band gives "
      f"{np.nanmedian(rel_d):.1%}.")
print("  The error is dominated by INDEPENDENT secondary uncertainty scattering account losses about")
print("  the cell's conditional mean - the term 7.2 says is discarded, which no banding removes.")
print("  The blur diagnostic finds cells where share MEANS vary; the dominant error is variance")
print("  ABOUT the mean. Refine bands only after confirming the reducible part is material.")

## Summary

| document section | here | note |
|---|---|---|
| §10.1–10.2 chunking, CRN | §3, §5 | fixed chunk grid; worker count out of run identity |
| §4.1.1–4.1.2 aggregate, per-year metrics | §4 | parallel, `np.add.at` / `np.maximum.at` |
| §4.1.3 retention | §6 | union measured; row-ranked cut shown to miss years |
| §4.1.4–4.1.5, §5.1–5.3 tail + body | §7 | one pass; workers return sums, division once |
| §5.2 band keying | §2 | kept on row loss; edges made a priori to drop a barrier |
| §8 validation | §8, §12 | corrections 3, 4 + degenerate-banding probe |
| §4.1.6, §10.3 persist | §10 | sorted Parquet part files; `chunk_years` in the audit |
| §4.2 reconstruction, allocation | §11 | never exercised on the allocation path |
| §7.1 re-ranking limit | §13 | correction 5; guard armed; headroom measured |
| §9 sizing, battery | §14 | correction 6; 13-check battery |

**Corrections carried:** (1) `α ≥ q`; (2) population covariance; (3) test 1 is an implementation test;
(4) per-body-row error added; (5) §7.1 arithmetic (`s_C = 13/37`, multiplier 59/37); (6) sizing must
condition events/year on the retained set; (7) band resolution buys little against irreducible
within-cell scatter.

**Exact:** co-TVaR at `α ≥ q`, account ranking within tail years, `Σₐ co-TVaRₐ = TVaR`, per-account AAL.
**Approximate:** per-account detail in body years — which the allocation never reads.
**Forbidden:** any mix-changing factor on the compressed store.

**Revision 2 (code review):** R1 content-addressed severity hash; R2 audit pins generator + worker
source + library versions; R3 loud failure on unreconstructable cells; R4 full-column CRN check;
R5 measured gating of book-dependent asserts; R6 relative T2 tolerance; R7 event-vectorised
generator (RNG layout change, versioned); R8 integer cell codes; R9 initializer-shipped shared
state; R10 misc guards; R11 sum-to-1 claim for unweighted shares corrected (test 1 is the catcher).

**Revision 3 (envelope retention):** E1 `Low`/`Env` accumulators in pass 1; E2 retention of
`{Env ≥ τ}` with the one-line containment proof, price measured; E3 three-zone guard + exact
re-weighted reader from `A_TAIL` (bounds pinned in the audit); E4 sampled-w validation (interior /
corners / cone) against CRN truth, in the battery; E5 harness axes `envelope`, `event` cells,
`analytic` shares (declared trade on test 1).

### Design notes (E6): one deferred, one elevated

**Annual-sum middle tier for the body.** Shares support AAL and conditionally-unbiased occurrence
means, but not per-account body EP curves — every reconstructed row in a cell carries the same
account mix, collapsing per-account annual variance. If the downstream contract grows to
account-level return periods, the right tier-2 object is the body-years account×year annual-sum
matrix: exact for *every* annual-grain read including arbitrary re-weighting (§7.1’s re-ranking
limitation does not exist at annual grain, since `Σₐ wₐ Aₐ,ₜ` re-ranks correctly for any `w`),
losing only occurrence grain. Whether it fits is a sizing question the §9 machinery already answers.

**Next work item (elevated post-review): single-pass a-priori retention.** Not a deferral — a
**deployment blocker**. Production account PLTs arrive once through a priced RMS API: pass 2
cannot run, and the envelope makes the problem *harder*, because τ must exist before the stream.
The pass-0 contract:

1. **Declare W first** — the envelope is a generation-time contract; the width sweep prices it.
2. **Conservative thresholds a priori.** Base: analytic `VaR_q(Y)`, `VaR_q(X)`. Envelope: τ̂ ≤ τ
   with margin — uniform box: τ = w_lo·VaR_α(Y); per-account boxes: the analytic AEP of the
   w_lo-scaled book (same FFT, scaled ELTs). OEP is closed form,
   `P(annual max > x) = 1 − exp(−Λ·S̄(x))`; AEP needs `catlib`’s compound-Poisson FFT with the
   per-event common-shock severity.
3. **Stream once**, retaining rows for years crossing the conservative thresholds while
   accumulating exact `Y/X/Low/Env`.
4. **Trim at stream end** — exact τ is known once `Low` is complete; trim the superset to exact
   `R`. Under-retention is *detectable* ex post but repairable only by a priced re-request, so the
   margin is sized from the analytic tail slope — that sizing is the work item.

A lossless sorted+zstd columnar baseline belongs in the harness as the null hypothesis. (The
annual-sum middle tier stays deferred: independent review measured ≈ 2× compression on this book,
which does not buy a tier.)

---

# Appendix · Alternate pathways — a strategy harness

Every design decision in the scheme is a switch. This section makes them explicit so they can be
A/B tested rather than assumed, and evaluates each against the **same CRN truth** — without common
random numbers the comparison is uninterpretable (§8).

| axis | options | document reference |
|---|---|---|
| `q` retention margin | 0.975 / 0.99 | §9 reduction 1 |
| `retention_basis` | `union` / `aep` / `oep` | §9 reduction 2 — `oep` fails containment |
| `tail_grain` | `occurrence` / `year` | §9 reduction 3 |
| `retention_key` | `year` / `row` | §4.1.3 point of care 1 — `row` is the documented failure |
| `band_basis` | `apriori` / `quantile` / `single` / `event` | §5.2 + the barrier trade-off; `event` is correction 7's limit |
| `n_bands` | any | §8 test 3 |
| `share_estimator` | `weighted` / `unweighted` / `analytic` | §5.1 — `unweighted` is the documented failure; `analytic` is a declared trade (no fit, T1 approximate) |
| `envelope` | `box` / `none` | E1–E4 — the re-weighted-exactness contract and its measured price |

Four presets are **expected to be rejected**, included so the battery is shown to catch them
(the failure presets carry `envelope="none"` so each axis is isolated — the envelope's own AEP
guarantee would otherwise repair OEP-only and row-keyed containment):
`unweighted` breaks test 1 — and *only* test 1: its densified shares still sum to 1 exactly,
because per-row shares do (Σₐ sₐ,ᵣ = 1; see R11) — `row-keyed` breaks containment and test 2,
and `oep`-only breaks containment — the OEP tail does not contain the AEP tail, which is why §4.1.3
takes the union. The fourth, `analytic` ELT shares, is not a failure mode but a **declared trade**:
zero fitting and zero barrier in exchange for the exact-AAL contract, so strict test 1 rejects it.

The harness reuses one generation pass rather than regenerating per strategy — it is a bench, not
the production path. The chunked two-pass production implementation is §3–§7 above.

In [ ]:
from dataclasses import dataclass, replace, asdict
import tempfile

@dataclass(frozen=True)
class Strategy:
    name:            str
    q:               float = 0.975        # retention percentile
    retention_basis: str   = "union"      # union | aep | oep
    retention_key:   str   = "year"       # year | row      (row = documented failure mode)
    tail_grain:      str   = "occurrence" # occurrence | year
    band_basis:      str   = "apriori"    # apriori | quantile | single | event
    n_bands:         int   = 8
    share_estimator: str   = "weighted"   # weighted | unweighted | analytic
    envelope:        str   = "box"        # box | none   (E1-E4 re-weighting contract)


def select_retention(st):
    kk = int(np.ceil((1 - st.q) * T_YEARS))
    if st.retention_key == "row":                       # 4.1.3: the documented mistake
        order = np.argsort(-portfolio_plt["loss"].to_numpy(), kind="stable")
        R_ = np.unique(portfolio_plt["year_id"].to_numpy()[order[:kk]])
    else:
        aep, oep = top_k_years(Y, kk), top_k_years(X, kk)
        R_ = {"aep": aep, "oep": oep}.get(st.retention_basis, np.union1d(aep, oep))
    if st.envelope == "box":                            # E2: one union, price measured below
        R_ = np.union1d(R_, R_ENV)
    return np.sort(R_)


def make_band_edges(st, body_port_=None):
    if st.band_basis in ("single", "event"):
        return {p: np.array([-np.inf, np.inf]) for p in BAND_EDGES} if st.band_basis == "single" \
               else "EVENT"                              # sentinel: cell = event_id
    if st.band_basis == "apriori":                       # no global barrier
        out = {}
        for pg, sub in catalogue.groupby("peril_group"):
            v = np.array([ev_mean[int(e)] for e in sub.event_id])
            out[pg] = np.concatenate([[-np.inf],
                                      np.geomspace(v.min(), v.max(), st.n_bands - 1), [np.inf]])
        return out
    out = {}                                             # quantile: needs the body -> barrier
    for pg, sub in body_port_.groupby("peril_group"):
        e = np.quantile(sub["loss"], np.linspace(0, 1, st.n_bands + 1))
        e[0], e[-1] = -np.inf, np.inf
        out[pg] = np.unique(e)
    return out


# correction 7 pushed to its limit: with one cell per EVENT, the a-priori share is
# derivable from the ELTs alone (normalised MeanLoss) - zero fitting, zero barrier.
# The trade: aggregate-then-divide exactness of per-account AAL needs shares FITTED
# on the body, so the analytic variant gives up test 1 (Monte-Carlo-scale error).
EV_ACCT_MEAN = (pd.concat([e.assign(accnt_no=a) for a, e in sorted(ELTS.items())])
                .pivot_table(index="EventId", columns="accnt_no", values="MeanLoss",
                             aggfunc="sum", fill_value=0.0))
SM_ANALYTIC  = (EV_ACCT_MEAN.div(EV_ACCT_MEAN.sum(axis=1), axis=0)
                .reindex(columns=ACCOUNTS).fillna(0.0))
SM_ANALYTIC.index = SM_ANALYTIC.index.astype(np.int64)


def fit_shares(st, bacct, bport):
    if st.share_estimator == "analytic":                 # a priori from the ELTs; no fit
        assert st.band_basis == "event", "analytic shares are defined per event"
        return SM_ANALYTIC
    num = bacct.groupby(["cell", "accnt_no"])["loss"].sum()
    if st.share_estimator == "weighted":                 # aggregate, THEN divide (5.1)
        s = num / bport.groupby("cell")["loss"].sum()
    else:                                                # unweighted mean of per-row ratios
        s_sum = bacct.groupby(["cell", "accnt_no"])["s"].sum()
        n_c   = bport.groupby("cell").size()
        s = pd.Series(s_sum.to_numpy()
                      / n_c.reindex(s_sum.index.get_level_values("cell")).to_numpy(),
                      index=s_sum.index)
    return (s.rename("share").reset_index()
             .pivot(index="cell", columns="accnt_no", values="share").fillna(0.0))

print("pathway functions defined: select_retention, make_band_edges, fit_shares "
      "(+ event cells, analytic shares, envelope axis)")

In [ ]:
def run_strategy(st, alpha=ALPHA):
    t0   = time.time()
    R_   = select_retention(st)
    inR  = np.zeros(T_YEARS, bool); inR[R_] = True

    # --- tail, at the requested grain -------------------------------------
    tail = account_plt_TRUE[inR[account_plt_TRUE["year_id"].to_numpy()]]
    store = (tail.groupby(["year_id", "accnt_no"], as_index=False)["loss"].sum()
             if st.tail_grain == "year" else tail)

    # --- body: retention first, body as the residual (5.3) ----------------
    bport = portfolio_plt[~inR[portfolio_plt["year_id"].to_numpy()]].copy()
    bport["peril_group"] = bport["event_id"].map(PERIL_OF)
    edges = make_band_edges(st, bport)
    bport["cell"] = (bport["event_id"].to_numpy() if isinstance(edges, str)   # event cells
                     else cat_worker.assign_cell(bport, edges, PERIL_CODE))
    bacct = (account_plt_TRUE[~inR[account_plt_TRUE["year_id"].to_numpy()]]
             .merge(bport[OCC_KEY + ["cell", "loss"]].rename(columns={"loss": "L_r"}),
                    on=OCC_KEY, how="left"))
    bacct["s"] = bacct["loss"] / bacct["L_r"]
    sm = fit_shares(st, bacct, bport)

    # --- containment (CORRECTION 1) ---------------------------------------
    m_     = int(np.ceil((1 - alpha) * T_YEARS))
    tail_y = top_k_years(Y, m_)
    contained = bool(inR[tail_y].all())

    # --- TEST 2: co-TVaR against truth ------------------------------------
    truth_tv = (account_plt_TRUE[np.isin(account_plt_TRUE["year_id"], tail_y)]
                .groupby(["year_id", "accnt_no"])["loss"].sum().unstack(fill_value=0.0)
                .reindex(index=tail_y, columns=ACCOUNTS).fillna(0.0).mean(axis=0))
    sel = store[np.isin(store["year_id"], tail_y)]
    got = (sel.groupby(["year_id", "accnt_no"])["loss"].sum().unstack(fill_value=0.0)
              .reindex(index=tail_y, columns=ACCOUNTS).fillna(0.0).mean(axis=0))
    e2  = float((got - truth_tv).abs().max())

    # --- TEST 2w (E4): re-weighted co-TVaR over the shared W samples ------
    A_st = (store.groupby(["year_id", "accnt_no"])["loss"].sum().unstack(fill_value=0.0)
            .reindex(columns=ACCOUNTS).fillna(0.0))
    yrsR, Amat = A_st.index.to_numpy(), A_st.to_numpy()
    e2w, cont_w = 0.0, True
    for tw, truth_w, w_ in zip(TAILS_W, TRUTH_W, W_SAMPLES):
        if not inR[tw].all():
            cont_w = False
            continue                                    # containment failed: exactness moot
        wv    = w_.reindex(ACCOUNTS).to_numpy()
        Yw_R  = Amat @ wv
        s_    = np.lexsort((yrsR, -Yw_R))[:m_alloc]
        alloc = (Amat[s_] * wv).mean(axis=0)
        e2w   = max(e2w, float(np.abs(alloc - truth_w.to_numpy()).max()
                               / np.abs(truth_w.to_numpy()).max()))
    reweight_ok = bool(cont_w and e2w < 1e-9)

    # --- TEST 1: per-account AAL ------------------------------------------
    rec = pd.DataFrame(sm.reindex(bport["cell"]).fillna(0.0).to_numpy()
                       * bport["loss"].to_numpy()[:, None],
                       columns=sm.columns, index=bport.index)
    aal_r = (store.groupby("accnt_no")["loss"].sum().reindex(ACCOUNTS).fillna(0.0)
             + rec.sum(axis=0).reindex(ACCOUNTS).fillna(0.0)) / T_YEARS
    e1 = float((aal_r - aal_truth).abs().max())

    # --- TEST 4: per-body-row error ---------------------------------------
    bt = (bacct.pivot_table(index=OCC_KEY, columns="accnt_no", values="loss",
                            aggfunc="sum", fill_value=0.0).reindex(columns=ACCOUNTS).fillna(0.0))
    bh = (rec.set_index(pd.MultiIndex.from_frame(bport[OCC_KEY]))
             .reindex(bt.index).reindex(columns=ACCOUNTS).fillna(0.0))
    dn = bt.to_numpy().sum(axis=1)
    e4 = float(np.nanmedian(np.abs(bh.to_numpy() - bt.to_numpy()).sum(axis=1)
                            / np.where(dn > 0, dn, np.nan)))

    with tempfile.TemporaryDirectory() as tmp:
        f = Path(tmp) / "t.parquet"
        store.sort_values(["year_id"]).to_parquet(f, index=False, compression="zstd")
        mb = f.stat().st_size / 1e6

    return {"strategy": st.name, "years": len(R_), "tail rows": len(store), "store MB": mb,
            "cells": len(sm), "shares sum 1": bool(np.allclose(sm.sum(axis=1), 1.0)),
            "contained": contained, "T1 AAL err": e1, "T2 coTVaR err": e2,
            "T2w reweight err": e2w if cont_w else np.nan, "reweight ok": reweight_ok,
            "T4 body median": e4, "secs": time.time() - t0}


PRESETS = [
    Strategy("baseline (q=.975, union, 8 apriori)"),
    Strategy("no envelope (rev-2 baseline)",        envelope="none"),
    Strategy("q=0.99 (tighter margin)",             q=0.99),
    Strategy("q=0.99, no envelope (escapes)",       q=0.99, envelope="none"),
    Strategy("AEP only (drop OEP union)",           retention_basis="aep"),
    Strategy("FAIL: OEP only (alloc is AEP)",       retention_basis="oep", envelope="none"),
    Strategy("tail at YEAR grain",                  tail_grain="year"),
    Strategy("quantile bands (adds a barrier)",     band_basis="quantile"),
    Strategy("32 bands",                            n_bands=32),
    Strategy("1 band per peril",                    band_basis="single"),
    Strategy("event cells (correction-7 limit)",    band_basis="event"),
    Strategy("TRADE: analytic ELT shares",          band_basis="event", share_estimator="analytic"),
    Strategy("FAIL: unweighted shares",             share_estimator="unweighted", envelope="none"),
    Strategy("FAIL: row-keyed retention",           retention_key="row", envelope="none"),
]
# failure presets carry envelope="none" so each axis is tested in isolation - the
# envelope's own AEP guarantee would otherwise repair OEP-only/row-keyed containment.
EXPECT_FAIL = {"FAIL: unweighted shares", "FAIL: row-keyed retention",
               "FAIL: OEP only (alloc is AEP)", "TRADE: analytic ELT shares"}

grid = pd.DataFrame([run_strategy(s) for s in PRESETS]).set_index("strategy")
display(grid)

In [ ]:
tol1 = 1e-9 * aal_truth.max()
tol2 = 1e-9 * tv_truth.max()   # R6: relative to the co-TVaR scale
NAME2ST = {s.name: s for s in PRESETS}
verdict = []
for name, r in grid.iterrows():
    ok = (r["contained"] and r["T1 AAL err"] < tol1 and r["T2 coTVaR err"] < tol2
          and r["shares sum 1"])
    if NAME2ST[name].envelope == "box":                 # E4: strategies CLAIMING the
        ok = ok and r["reweight ok"]                    # contract must also honour it
    verdict.append("USABLE" if ok else "REJECTED")
grid["verdict"] = verdict
display(grid[["contained", "shares sum 1", "T1 AAL err", "T2 coTVaR err",
              "T2w reweight err", "reweight ok", "verdict"]])

# R5: each expected rejection is a BOOK PROPERTY or a declared trade, not an identity -
# measure it before asserting it. Section 6's own rule: measure, do not assume.
m_a = int(np.ceil((1 - ALPHA) * T_YEARS))
def expected_to_fail(st):
    if st.share_estimator == "unweighted":
        return True    # test-1 error = n*Cov(s, L) per cell; vanishing in EVERY cell at
                       # once to machine precision is measure-zero with continuous losses
    if st.share_estimator == "analytic":
        return True    # declared trade: exact-AAL contract given up for zero fitting
    return len(np.setdiff1d(top_k_years(Y, m_a), select_retention(st))) > 0

for st in PRESETS:
    if st.name not in EXPECT_FAIL:
        assert grid.loc[st.name, "verdict"] == "USABLE", f"{st.name} unexpectedly rejected"
    elif expected_to_fail(st):
        assert grid.loc[st.name, "verdict"] == "REJECTED", f"{st.name} should have been caught"
    else:
        print(f"note: this book happens to satisfy containment for '{st.name}' - the "
              "failure mode is book-dependent and was measured absent, not assumed (R5)")
print("all failure modes caught where the book exhibits them; every legitimate pathway passes OK")
print()
print("FINDING - section 9 reduction 2 is not symmetric. Dropping to AEP-only is safe for an")
print("  AEP-basis allocation; dropping to OEP-only is NOT: the top-k years by largest single")
print("  occurrence do not contain the top-k by annual total. The union exists for this reason,")
print("  and the containment check is what catches it.")

# --- what the envelope buys, measured (E1-E4) ------------------------------
base  = grid.loc["baseline (q=.975, union, 8 apriori)"]
noenv = grid.loc["no envelope (rev-2 baseline)"]
print()
print(f"ENVELOPE PRICE vs CAPABILITY: +{base['tail rows']-noenv['tail rows']:,.0f} tail rows "
      f"(+{100*(base['tail rows']/noenv['tail rows']-1):.0f}%), "
      f"+{base['store MB']-noenv['store MB']:.3f} MB")
if not noenv["reweight ok"]:
    print(f"  without it, re-weighted allocation is NOT exact: max rel err "
          f"{noenv['T2w reweight err']:.2e} over the sampled W"
          if np.isfinite(noenv["T2w reweight err"]) else
          "  without it, some sampled-w tails ESCAPE retention entirely - silently wrong reads")
else:
    print("  note: at this book/box the rev-2 margin covered every sampled w (measured,")
    print("  book-dependent - R5); the envelope converts that coincidence into a guarantee")

# the zero-headroom pairing: at q = alpha the base margin has nothing to spare, so any
# coverage of re-weighted tails is pure coincidence - measure it (R5)
st99n = NAME2ST["q=0.99, no envelope (escapes)"]
R99n  = select_retention(st99n)
esc99 = [int(len(np.setdiff1d(tw, R99n))) for tw in TAILS_W]
if max(esc99) > 0:
    print(f"  ZERO-HEADROOM PAIR (q=0.99): without the envelope, {min(esc99)}-{max(esc99)} of "
          f"{m_alloc:,} true tail years per sampled w")
    print("  escape retention entirely - a silently wrong re-weighted read. With it: exact.")
    print("  The guarantee is doing measurable work exactly where the margin runs out.")
else:
    print("  note: even at q=0.99 this book's sampled w stayed covered (measured - R5)")

print()
print("SIZE REDUCTION, against section 9's recommended order:")
for name in ["q=0.99 (tighter margin)", "AEP only (drop OEP union)", "tail at YEAR grain"]:
    r = grid.loc[name]
    print(f"  {name:32s} {r['tail rows']:7,.0f} rows ({100*r['tail rows']/base['tail rows']:5.1f}% "
          f"of baseline), {r['store MB']:.3f} MB, alpha>=q {'OK' if r['contained'] else 'BROKEN'}")
# --- post-review FINDING: the envelope disables section 9 reductions 1 and 2 ---
k99_  = int(np.ceil((1 - 0.99) * T_YEARS))
k975_ = int(np.ceil((1 - Q_RETAIN) * T_YEARS))
coll1 = np.array_equal(np.sort(np.union1d(np.union1d(top_k_years(Y, k99_),
                                                     top_k_years(X, k99_)), R_ENV)), np.sort(R_ENV))
coll2 = np.array_equal(np.sort(np.union1d(top_k_years(Y, k975_), R_ENV)), np.sort(R_ENV))
print()
print("FINDING - the envelope disables section 9 reductions 1 and 2. tau is set from ALPHA, not")
print("  q, so the envelope floor |R_ENV| is invariant to tightening q and to dropping the OEP")
print(f"  union: at this book q=0.99-union-envelope == R_ENV as a SET IDENTITY ({coll1}), and")
print(f"  AEP-only-union-envelope == R_ENV as a set identity ({coll2}). Once W is contracted,")
print("  GRAIN is the only remaining size lever; the sizing formula's retention/k factor")
print("  (incl. envelope) is the number that carries this.")

# --- W-width price sweep (closed form for uniform boxes: only spread = hi/lo matters:
#     retain {Y >= Y_(m)/s} union {X >= X_(m)/s} union base). The envelope is a
#     GENERATION-TIME contract pinned in the audit, not a read-time knob - per-account
#     boxes need the streamed accumulators (E1) - so price W from this curve BEFORE
#     committing storage: change W_LO/W_HI in cell 1 and re-run.
Ym_, Xm_ = np.partition(Y, -m_alloc)[-m_alloc], np.partition(X, -m_alloc)[-m_alloc]
rpy = (account_plt_TRUE["year_id"].value_counts()
       .reindex(np.arange(T_YEARS)).fillna(0).astype(int).to_numpy())
print()
print("W-WIDTH PRICE SWEEP (uniform boxes, closed form):")
prev = -1
for s_ in [1.0, 1.2, W_HI / W_LO, 2.0, 3.0, 4.0]:
    Ru_ = np.union1d(R_BASE, np.flatnonzero((Y >= Ym_ / s_) | (X >= Xm_ / s_)))
    r_  = int(rpy[Ru_].sum())
    assert len(Ru_) >= prev, "sweep must be monotone in spread"
    prev = len(Ru_)
    tag = "  <- contracted W" if abs(s_ - W_HI / W_LO) < 1e-9 else ""
    print(f"  spread {s_:5.3f}: {len(Ru_):6,} years ({100*len(Ru_)/T_YEARS:5.2f}%), "
          f"{r_:8,} tail rows ({r_/base['tail rows']:5.2f}x baseline){tag}")
print("  plateau below ~1.4 (the q-margin already covers), knee near 2, cliff beyond 3:")
print("  the cost is violently non-linear in spread - price the contract before generation.")

print()
print("BAND RESOLUTION (correction 7 - little leverage, taken to its limit):")
for name in ["1 band per peril", "baseline (q=.975, union, 8 apriori)", "32 bands",
             "quantile bands (adds a barrier)", "event cells (correction-7 limit)",
             "TRADE: analytic ELT shares"]:
    print(f"  {name:36s} {grid.loc[name,'cells']:3.0f} cells, "
          f"T4 median {grid.loc[name,'T4 body median']:.1%}, "
          f"T1 {grid.loc[name,'T1 AAL err']:.1e}")

### Reading the harness

- **Legitimate pathways are interchangeable on the allocation.** Every non-failure preset gives
  identical co-TVaR and per-account AAL. They differ only in footprint and in body fidelity, which
  the allocation never reads — so choose on storage, not on accuracy.
- **The §9 reduction order is testable.** Run the three size reductions and read the rows/MB column
  rather than accepting the ordering on faith; which one wins is book-dependent.
- **Both documented failure modes are caught, by different tests.** Unweighted shares break test 1 —
  and only test 1: their densified shares still sum to 1 exactly, since per-row shares do (R11),
  so the sum-to-1 invariant *cannot* catch them; row-keyed retention breaks containment and test 2. That is the
  separation of concerns corrections 3 and 4 were arguing for — one test per defect.
- **The `envelope` axis prices the contract.** Compare baseline vs `no envelope`: the rows/MB
  delta is what exact re-weighting over `cone(W)` costs, and the `T2w`/`reweight ok` columns show
  what it buys — measured, not assumed (R5): on some books the base margin covers the sampled `w`
  by coincidence; the envelope converts coincidence into guarantee. The `q=0.99` pair shows the
  other side: with zero margin headroom the sampled-`w` tails escape retention without it. The
  closed-form width sweep prices candidate contracts before generation — the envelope is a
  generation-time contract pinned in the audit, not a read-time knob, which is the design answer
  to “sweep the width as a preset”: change `W_LO/W_HI` in cell 1 and re-run.
- **`event` cells are correction 7's endpoint.** The finest partition costs nothing extra to fit
  (aggregate-then-divide is exact for any partition) and bounds what banding can ever recover;
  `analytic` shares then remove the fit entirely at the declared price of test 1.
- **`quantile` vs `apriori` bands** is the barrier trade-off made concrete: compare the `cells` and
  `T4` columns, and decide whether the difference justifies a second global synchronisation point.

To sweep an axis not in the preset list, use `replace()`: e.g.
`grid2 = pd.DataFrame([run_strategy(replace(PRESETS[0], name=f"{n} bands", n_bands=n))
                       for n in [2, 4, 8, 16, 32, 64]]).set_index("strategy")`.